# Original NF4 tiled register-reuse — MICROBENCH_ONLY

One materially different fixed64x64/4-warp design after original16x8 v4 lost all six warm medians. Each warp owns32x32, reusing each A fragment across4 column fragments and each decoded B across2 row fragments. At M2048/N1536 this reduces CTAs32x, source-level A requests4x, packed/decode requests2x; these are not measured DRAM or speed claims. Original single-warp reference remains frozen in this packet only for preflight. No vendor kernel copied, autotuning, loop-unroll-only tuning, shared-memory synchronization or GPU-specific runtime changes.

Before timing, target preflight demands12 exact comparisons against frozen original-MMA outputs, exact native decoded FP16 weights, unchanged native engineering tolerance atol.003/rtol.002, original8-instruction SM75 MMA128-thread lane layout/no-FMA gate, no compiler-reported spills/shared memory,6 exact native-gradient cases, nonfinite classes, the source-bound staged-bias gate, and12 real native Linear4bit/Parameter activation/gradient/RNG/identity cases. Target register resources are exported. All old math and Parameter/autograd ownership remain unchanged; the only additional input boundary is4-byte alignment for raw packed half2 activation loads. Misaligned calls fall back unchanged.

The earlier actual sparse T4 diagnostic observed native NF4/F.linear1.0 and candidate FP32-add-before-half1.0009765625. Exact candidate intended arithmetic plus unchanged native closeness remains the disclosed gate; no hidden native accumulator behavior is assumed. Synthetic module inputs/timing shapes are not captured fixed-task q/v activations. No pretrained model, training, installer, score, admission, share or whole-model memory claim.

Native then candidate timing uses fresh workers/private empty caches, isolated from preflight/reference compilation, six fixed M65/532/2048,N256/1536,K1536 shapes. Cold wrapper wall includes JIT/setup; warm CUDA-event intervals include dispatch/launch gaps. Each GPU child <=80s; service caps authoritative. Protected runtime/source/process-tree/lock/quarantine/idle/export helpers unchanged. No downloads, installs, credentials, cap changes or automatic retry. Parent alone submits after independent review.


In [ ]:
import os,base64,hashlib,tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS')
assert Path('/kaggle/working').is_dir()
PAYLOAD = {'solution/__init__.py': 'IiIiUmVzZWFyY2gtb25seSBndWFyZGVkIGZvcndhcmQgaGVscGVyLiBObyB0YXNrIGluc3RhbGxhdGlvbiBlbnRyeXBvaW50LiIiIgo=', 'solution/bias_gate_provenance.json': 'ewogICJzdGF0dXMiOiAiQUNUVUFMX1RBUkdFVF9ESUFHTk9TVElDX0JPVU5EIiwKICAiam9iX2lkIjogInJlZGFjdGVkLWpvYiIsCiAgImFjY2VwdGVkX3NvdXJjZV9zaGEyNTYiOiAiZmM4OGExYTJlYzgwYWI4YzQ4YjkzMzZhYzIyNTIyMTk1YTMwYTE5ODQ5MDBkNTE5ZmRjZWU5NDk2YTRiMzdjMyIsCiAgIm91dHB1dF96aXBfc2hhMjU2IjogIjVmMGU2ODIxNzU0Zjg5YTI5MmZjMTM2ZDNkZjM4NDA0YjFjZTk3NzcyMzEyOThiZWUzYzA0NjUwN2I0YTUxZjMiLAogICJkaWFnbm9zdGljX3JlcG9ydF9zaGEyNTYiOiAiMGQwYjUxOTBkMjE3NjBkZTg2NmEwNjAxNmIyY2NhYzEzOTg1MDkxYzRhOWM2MGI0NjNhOWYyZDdmZmVjODQ3ZCIsCiAgImZyb3plbl92M19rZXJuZWxfc2hhMjU2IjogImIwMzc5NWYxZTRmZWFhOTliNzhkZWUzZDFiOGY4YzFlYWNhMzc5NDNhODJlZGMzZWYxNDAwZTFjYTRjYzFjZjgiLAogICJvYnNlcnZlZF9uYXRpdmVfRmxpbmVhcl9hbmRfZGlzcGF0Y2giOiAxLjAsCiAgIm9ic2VydmVkX2NhbmRpZGF0ZV9hbmRfZnAzMl9hZGRfdGhlbl9oYWxmIjogMS4wMDA5NzY1NjI1LAogICJkZWNvZGVkX2ZwMTZfd2VpZ2h0c19leGFjdCI6IHRydWUsCiAgIm5hdGl2ZV9pbnRlcm5hbF9hY2N1bXVsYXRvcl9ub3Rfb2JzZXJ2ZWQiOiB0cnVlLAogICJpbmZlcmVuY2Vfc2NvcGUiOiAiT25lIGV4YWN0IHNwYXJzZSB0YXJnZXQgd2l0bmVzcyBkaXNwcm92ZXMgcHJldmlvdXMgcmVxdWlyZWQgZXhhY3QtbmF0aXZlIEZQMzItYmlhcy1zdGFnZSBhc3N1bXB0aW9uOyBkb2VzIG5vdCBpZGVudGlmeSBldmVyeSBuYXRpdmUgYWNjdW11bGF0aW9uIHN0YWdlLiIsCiAgInJldmlzZWRfZ2F0ZSI6ICJLZXJuZWwgdW5jaGFuZ2VkLiBFeGFjdCBjYW5kaWRhdGUgaW50ZW5kZWQgRlAzMi1hZGQtYmVmb3JlLWhhbGYgc3RhZ2U7IHNhbWUgcHJlZGVjbGFyZWQgbmF0aXZlIGVuZ2luZWVyaW5nX2Nsb3NlIGF0b2wgLjAwMyBydG9sIC4wMDIuIE5hdGl2ZSBlcXVhbGl0eS9zdGFnZWQgY29tcGFyaXNvbnMgZXhwb3J0ZWQgYXMgb2JzZXJ2YXRpb25zLiIsCiAgIm9sZF92M19leGFjdF9uYXRpdmVfZmFpbHVyZV9wcmVzZXJ2ZWQiOiB0cnVlCn0K', 'solution/bias_stage_gate.py': 'IiIiRXhwbGljaXQgY2FuZGlkYXRlIGFyaXRobWV0aWMgc3RhZ2UsIHNlcGFyYXRlIGZyb20gbmF0aXZlIGFwcHJveGltYXRpb24uCgpUaGUgbmF0aXZlIENVREEgc3BhcnNlIHdpdG5lc3MgZGlzcHJvdmVkIGFuIGVhcmxpZXIgQ1BVLWRlcml2ZWQgYXNzdW1wdGlvbi4KTm8gbmF0aXZlIGhpZGRlbiBhY2N1bXVsYXRvciBpcyBleHBvc2VkIG9yIGluZmVycmVkIGhlcmUuIFRvbGVyYW5jZXMgYXJlIHRoZQp1bmNoYW5nZWQgcHJlZGVjbGFyZWQgZW5naW5lZXJpbmcgZ2F0ZSwgbm90IGEgZml4ZWQtdGFzayBxdWFsaXR5IGRlY2lzaW9uLgoiIiIKaW1wb3J0IHRvcmNoCmZyb20gLmZpeHR1cmVzIGltcG9ydCBleGFjdCwgZW5naW5lZXJpbmdfY2xvc2UKCgpkZWYgdmVyaWZ5X2JpYXNfc3RhZ2UoYWN0dWFsLCBuYXRpdmUsIGZwMzJfcmVmZXJlbmNlX3dpdGhvdXRfYmlhcywgYmlhcywKICAgICAgICAgICAgICAgICAgICAgIG5hdGl2ZV9tYXRlcmlhbGl6ZWRfd2l0aG91dF9iaWFzKToKICAgIGlmIGZwMzJfcmVmZXJlbmNlX3dpdGhvdXRfYmlhcy5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyIG9yIGJpYXMuZHR5cGUgIT0gdG9yY2guZmxvYXQxNjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0V4cGxpY2l0IHJlZmVyZW5jZSBzdGFnZSBkdHlwZSBtaXNtYXRjaCcpCiAgICBpbnRlbmRlZCA9IChmcDMyX3JlZmVyZW5jZV93aXRob3V0X2JpYXMgKyBiaWFzLmZsb2F0KCkpLmhhbGYoKQogICAgcG9zdHJvdW5kID0gZnAzMl9yZWZlcmVuY2Vfd2l0aG91dF9iaWFzLmhhbGYoKSArIGJpYXMKICAgIGlmIHRvcmNoLmVxdWFsKGludGVuZGVkLCBwb3N0cm91bmQpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2FuZGlkYXRlIHN0YWdlZC1iaWFzIG5lZ2F0aXZlIHdpdG5lc3MgYmVjYW1lIHZhY3VvdXMnKQogICAgIyBWZXJpZnkgdGhlIGNhbmRpZGF0ZSdzIGludGVuZGVkIGFyaXRobWV0aWMsIG5vdCBhbiBhc3N1bWVkIENVREEgZXBpbG9ndWUuCiAgICBleGFjdChhY3R1YWwsIGludGVuZGVkKQogICAgbnVtZXJpYyA9IGVuZ2luZWVyaW5nX2Nsb3NlKGFjdHVhbCwgbmF0aXZlKQogICAgcmV0dXJuIHsKICAgICAgICAnY2FuZGlkYXRlX2ZwMzJfYmlhc19iZWZvcmVfaGFsZl9leGFjdCc6IFRydWUsCiAgICAgICAgJ2NhbmRpZGF0ZV9wb3N0cm91bmRfbmVnYXRpdmVfd2l0bmVzc19ub252YWN1b3VzJzogVHJ1ZSwKICAgICAgICAnbmF0aXZlX2NvbXBhcmlzb24nOiBudW1lcmljLAogICAgICAgICdvYnNlcnZhdGlvbnMnOiB7CiAgICAgICAgICAgICduYXRpdmVfZXF1YWxzX2NhbmRpZGF0ZSc6IGJvb2wodG9yY2guZXF1YWwobmF0aXZlLCBhY3R1YWwpKSwKICAgICAgICAgICAgJ25hdGl2ZV9lcXVhbHNfZXhwbGljaXRfZnAzMl9iaWFzX2JlZm9yZV9oYWxmJzogYm9vbCh0b3JjaC5lcXVhbChuYXRpdmUsIGludGVuZGVkKSksCiAgICAgICAgICAgICduYXRpdmVfZXF1YWxzX21hdGVyaWFsaXplZF9oYWxmX3RoZW5fYmlhcyc6IGJvb2wodG9yY2guZXF1YWwobmF0aXZlLCBuYXRpdmVfbWF0ZXJpYWxpemVkX3dpdGhvdXRfYmlhcyArIGJpYXMpKSwKICAgICAgICAgICAgJ25hdGl2ZV91bmlxdWUnOiB0b3JjaC51bmlxdWUobmF0aXZlLmRldGFjaCgpKS5mbG9hdCgpLmNwdSgpLnRvbGlzdCgpLAogICAgICAgICAgICAnY2FuZGlkYXRlX3VuaXF1ZSc6IHRvcmNoLnVuaXF1ZShhY3R1YWwuZGV0YWNoKCkpLmZsb2F0KCkuY3B1KCkudG9saXN0KCksCiAgICAgICAgICAgICdmcDMyX3JlZmVyZW5jZV93aXRob3V0X2JpYXNfdW5pcXVlJzogdG9yY2gudW5pcXVlKGZwMzJfcmVmZXJlbmNlX3dpdGhvdXRfYmlhcy5kZXRhY2goKSkuY3B1KCkudG9saXN0KCksCiAgICAgICAgICAgICdpbnRlbmRlZF91bmlxdWUnOiB0b3JjaC51bmlxdWUoaW50ZW5kZWQuZGV0YWNoKCkpLmZsb2F0KCkuY3B1KCkudG9saXN0KCksCiAgICAgICAgICAgICdwb3N0cm91bmRfdW5pcXVlJzogdG9yY2gudW5pcXVlKHBvc3Ryb3VuZC5kZXRhY2goKSkuZmxvYXQoKS5jcHUoKS50b2xpc3QoKSwKICAgICAgICB9LAogICAgICAgICduYXRpdmVfaW50ZXJuYWxfYWNjdW11bGF0b3Jfb2JzZXJ2ZWQnOiBGYWxzZSwKICAgICAgICAndGFza19xdWFsaXR5JzogJ1VOS05PV04nLAogICAgfQo=', 'solution/fixtures.py': 'IiIiU21hbGwgc3ludGhldGljIHEvdi1zaGFwZWQgaW5wdXRzLCBub3QgY2FwdHVyZWQgdGFzayBhY3RpdmF0aW9ucy4iIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBqc29uCmltcG9ydCB0b3JjaAppbXBvcnQgYml0c2FuZGJ5dGVzIGFzIGJuYgpmcm9tIGJpdHNhbmRieXRlcyBpbXBvcnQgZnVuY3Rpb25hbCBhcyBGCgpFWFBFQ1RFRD1qc29uLmxvYWRzKChQYXRoKF9fZmlsZV9fKS53aXRoX25hbWUoJ25hdGl2ZV9zb3VyY2VfbWFuaWZlc3QuanNvbicpKS5yZWFkX3RleHQoKSkKU0hBUEVTPSgoNjUsMjU2KSwoNTMyLDI1NiksKDIwNDgsMjU2KSwoNjUsMTUzNiksKDUzMiwxNTM2KSwoMjA0OCwxNTM2KSkKCmRlZiBmaXh0dXJlKG0sbix3aXRoX2JpYXM9VHJ1ZSk6CiAgICB0b3JjaC5tYW51YWxfc2VlZCgxNzI5K24pCiAgICB3PXRvcmNoLnJhbmRuKG4sMTUzNixkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpKi4xCiAgICBwYWNrZWQsc3RhdGU9Ri5xdWFudGl6ZV80Yml0KHcsYmxvY2tzaXplPTY0LGNvbXByZXNzX3N0YXRpc3RpY3M9VHJ1ZSxxdWFudF90eXBlPSduZjQnKQogICAgYT10b3JjaC5yYW5kbihtLDE1MzYsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KSouMQogICAgYmlhcz10b3JjaC5yYW5kbihuLGRldmljZT0nY3VkYScsZHR5cGU9dG9yY2guZmxvYXQxNikqLjAzIGlmIHdpdGhfYmlhcyBlbHNlIE5vbmUKICAgIHJldHVybiBhLHBhY2tlZCxzdGF0ZSxiaWFzCgpkZWYgY2FsbChkYXRhKToKICAgIGEscGFja2VkLHN0YXRlLGJpYXM9ZGF0YQogICAgcmV0dXJuIGJuYi5tYXRtdWxfNGJpdChhLHBhY2tlZC50KCkscXVhbnRfc3RhdGU9c3RhdGUsYmlhcz1iaWFzKQoKZGVmIGNsYXNzZXMoYSxiKToKICAgIGlmIGEuc2hhcGUhPWIuc2hhcGUgb3IgYS5kdHlwZSE9Yi5kdHlwZTpyYWlzZSBSdW50aW1lRXJyb3IoJ1NoYXBlIG9yIGR0eXBlIG1pc21hdGNoJykKICAgIGZvciBmbiBpbiAodG9yY2guaXNuYW4sdG9yY2guaXNwb3NpbmYsdG9yY2guaXNuZWdpbmYpOgogICAgICAgIGlmIG5vdCB0b3JjaC5lcXVhbChmbihhKSxmbihiKSk6cmFpc2UgUnVudGltZUVycm9yKCdOb25maW5pdGUgY2xhc3NlcyBkaWZmZXInKQoKZGVmIGV4YWN0KGEsYik6CiAgICBjbGFzc2VzKGEsYikKICAgIGZpbml0ZT10b3JjaC5pc2Zpbml0ZShhKSZ0b3JjaC5pc2Zpbml0ZShiKQogICAgaWYgbm90IHRvcmNoLmVxdWFsKGFbZmluaXRlXSxiW2Zpbml0ZV0pOnJhaXNlIFJ1bnRpbWVFcnJvcignRmluaXRlIHZhbHVlcyBkaWZmZXInKQogICAgaWYgbm90IHRvcmNoLmVxdWFsKHRvcmNoLnNpZ25iaXQoYVthPT0wXSksdG9yY2guc2lnbmJpdChiW2I9PTBdKSk6cmFpc2UgUnVudGltZUVycm9yKCdTaWduZWQgemVyb3MgZGlmZmVyJykKCmRlZiBlbmdpbmVlcmluZ19jbG9zZShhLGIpOgogICAgY2xhc3NlcyhhLGIpCiAgICBmaW5pdGU9dG9yY2guaXNmaW5pdGUoYSkmdG9yY2guaXNmaW5pdGUoYikKICAgIHRvcmNoLnRlc3RpbmcuYXNzZXJ0X2Nsb3NlKGFbZmluaXRlXSxiW2Zpbml0ZV0sYXRvbD0uMDAzLHJ0b2w9LjAwMikKICAgIGRlbHRhPShhW2Zpbml0ZV0uZmxvYXQoKS1iW2Zpbml0ZV0uZmxvYXQoKSkuYWJzKCkKICAgIHJldHVybiB7J21heF9hYnNfZXJyb3InOmZsb2F0KGRlbHRhLm1heCgpLml0ZW0oKSkgaWYgZGVsdGEubnVtZWwoKSBlbHNlIDAuLAogICAgICAgICAgICAnYXRvbCc6LjAwMywncnRvbCc6LjAwMiwnY3JpdGVyaW9uJzonZml4ZWQgc3ludGhldGljIGVuZ2luZWVyaW5nIGdhdGUsIG5vdCB0YXNrIHF1YWxpdHknfQo=', 'solution/guarded_forward.py': 'IiIiUmVzZWFyY2gtb25seSBmYWxsYmFjayByZXBsYWNlbWVudDsgbmF0aXZlIGRpc3BhdGNoZXIvYXV0b2dyYWQgc3RheSBvd25lZC4KCk5vIHBlcnNpc3RlbnQgaW5zdGFsbGF0aW9uIGFuZCBubyBjdXN0b20gYmFja3dhcmQuIERpcmVjdCBncmFkLWVuYWJsZWQgdXNlIGlzCmluZWxpZ2libGU6IG9ubHkgbmF0aXZlIE1hdE11bDRCaXQuZm9yd2FyZCdzIG5vLWdyYWQgY29udGV4dCBtYXkgcm91dGUgdHJhaW5hYmxlCmlucHV0cyBoZXJlLiBUaGlzIGhlbHBlciBpcyBub3QgYW4gYWRtaXNzaW9uIG9yIGZ1bGwtdGFzayBjYW5kaWRhdGUuCiIiIgpmcm9tIGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyCmltcG9ydCBoYXNobGliCmltcG9ydCBpbnNwZWN0CmltcG9ydCBtYXRoCmltcG9ydCB0b3JjaAppbXBvcnQgdHJpdG9uCmZyb20gLmtlcm5lbCBpbXBvcnQgc291cF9uZjRfbW1hX3RpbGVkX3JldXNlCmZyb20gLnBhcmFtZXRlcl9iaWFzIGltcG9ydCBub3JtYWxpemVkX25hdGl2ZV9iaWFzCgpDT1VOVFMgPSB7ImN1c3RvbSI6IDAsICJuYXRpdmUiOiAwfQpDT01QSUxFRCA9IHt9CkZBTExCQUNLX1JFQVNPTlMgPSB7fQoKCmRlZiByZWFzb24oQSwgQiwgc2hhcGVCLCBhYnNtYXgsIGJsb2Nrc2l6ZSwgcXVhbnRfdHlwZSwgYmlhcz1Ob25lLAogICAgICAgICAgIGFic21heF84Yml0PU5vbmUsIGFic21heF9jb2RlPU5vbmUsIGFic21heF9vZmZzZXQ9Tm9uZSk6CiAgICBpZiB0eXBlKGJsb2Nrc2l6ZSkgaXMgbm90IGludCBvciBibG9ja3NpemUgIT0gNjQgb3IgcXVhbnRfdHlwZSAhPSAibmY0IjoKICAgICAgICByZXR1cm4gImZvcm1hdCIKICAgIGlmIHR5cGUoc2hhcGVCKSBub3QgaW4gKGxpc3QsIHR1cGxlLCB0b3JjaC5TaXplKSBvciBsZW4oc2hhcGVCKSAhPSAyOgogICAgICAgIHJldHVybiAic2hhcGVfbWV0YWRhdGEiCiAgICBpZiBhbnkodHlwZSh4KSBpcyBub3QgaW50IGZvciB4IGluIHNoYXBlQikgb3IgdHVwbGUoc2hhcGVCKSBub3QgaW4gKCgxNTM2LDE1MzYpLCgyNTYsMTUzNikpOgogICAgICAgIHJldHVybiAic2hhcGVfc2NvcGUiCiAgICBuLGsgPSBzaGFwZUIKICAgIG9wZXJhbmRzID0gKEEsQixhYnNtYXgsYWJzbWF4XzhiaXQsYWJzbWF4X2NvZGUsYWJzbWF4X29mZnNldCkKICAgIGlmIGFueSh0eXBlKHgpIGlzIG5vdCB0b3JjaC5UZW5zb3IgZm9yIHggaW4gb3BlcmFuZHMpIG9yIChiaWFzIGlzIG5vdCBOb25lIGFuZCB0eXBlKGJpYXMpIGlzIG5vdCB0b3JjaC5UZW5zb3IpOgogICAgICAgIHJldHVybiAidGVuc29yX3R5cGUiCiAgICBpZiBBLm5kaW0gbm90IGluICgyLDMpIG9yIEEuc2hhcGVbLTFdICE9IGsgb3IgKEEubmRpbSA9PSAzIGFuZCBBLnNoYXBlWzBdICE9IDQpOgogICAgICAgIHJldHVybiAiYWN0aXZhdGlvbl9nZW9tZXRyeSIKICAgIG0gPSBtYXRoLnByb2QoQS5zaGFwZVs6LTFdKQogICAgaWYgbm90IDY1IDw9IG0gPD0gMjA0ODoKICAgICAgICByZXR1cm4gInJvd19zY29wZSIKICAgIGFsbF90ZW5zb3JzID0gb3BlcmFuZHMgKyAoKCkgaWYgYmlhcyBpcyBOb25lIGVsc2UgKGJpYXMsKSkKICAgIGlmIGFueSh4LmlzX25lZygpIG9yIHguaXNfY29uaigpIG9yIG5vdCB4LmlzX2NvbnRpZ3VvdXMoKSBmb3IgeCBpbiBhbGxfdGVuc29ycyk6CiAgICAgICAgcmV0dXJuICJsYXp5X29yX3N0cmlkZWQiCiAgICBpZiBhbnkoeC5kZXZpY2UgIT0gQS5kZXZpY2UgZm9yIHggaW4gYWxsX3RlbnNvcnMpIG9yIEEuZGV2aWNlLnR5cGUgIT0gImN1ZGEiOgogICAgICAgIHJldHVybiAiZGV2aWNlIgogICAgaWYgQS5kdHlwZSAhPSB0b3JjaC5mbG9hdDE2IG9yIEIuZHR5cGUgIT0gdG9yY2gudWludDggb3IgQi5udW1lbCgpICE9IG4qay8vMiBvciBCLm5kaW0gbm90IGluICgxLDIpOgogICAgICAgIHJldHVybiAiYWN0aXZhdGlvbl9vcl9wYWNrZWRfZHR5cGVfZ2VvbWV0cnkiCiAgICBpZiBBLmRhdGFfcHRyKCklNDoKICAgICAgICByZXR1cm4gImFjdGl2YXRpb25faGFsZjJfYWxpZ25tZW50IgogICAgYmxvY2tzPW4qay8vNjQKICAgIGlmIChhYnNtYXhfOGJpdC5kdHlwZSAhPSB0b3JjaC51aW50OCBvciBhYnNtYXhfOGJpdC5uZGltICE9IDEgb3IgYWJzbWF4XzhiaXQubnVtZWwoKSAhPSBibG9ja3MKICAgICAgICAgICAgb3IgYWJzbWF4LmR0eXBlICE9IHRvcmNoLmZsb2F0MzIgb3IgYWJzbWF4Lm5kaW0gIT0gMSBvciBhYnNtYXgubnVtZWwoKSAhPSB0cml0b24uY2RpdihibG9ja3MsMjU2KQogICAgICAgICAgICBvciBhYnNtYXhfY29kZS5kdHlwZSAhPSB0b3JjaC5mbG9hdDMyIG9yIGFic21heF9jb2RlLm5kaW0gIT0gMSBvciBhYnNtYXhfY29kZS5udW1lbCgpICE9IDI1NgogICAgICAgICAgICBvciBhYnNtYXhfb2Zmc2V0LmR0eXBlICE9IHRvcmNoLmZsb2F0MzIgb3IgYWJzbWF4X29mZnNldC5udW1lbCgpICE9IDEpOgogICAgICAgIHJldHVybiAibmVzdGVkX3NjYWxlX2dlb21ldHJ5IgogICAgaWYgYW55KHgucmVxdWlyZXNfZ3JhZCBmb3IgeCBpbiBvcGVyYW5kc1sxOl0pOgogICAgICAgIHJldHVybiAiZGlmZmVyZW50aWFibGVfbWV0YWRhdGEiCiAgICBpZiBiaWFzIGlzIG5vdCBOb25lIGFuZCAoYmlhcy5kdHlwZSAhPSB0b3JjaC5mbG9hdDE2IG9yIGJpYXMuc2hhcGUgIT0gKG4sKSk6CiAgICAgICAgcmV0dXJuICJiaWFzX2dlb21ldHJ5IgogICAgaWYgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCkgYW5kIChBLnJlcXVpcmVzX2dyYWQgb3IgKGJpYXMgaXMgbm90IE5vbmUgYW5kIGJpYXMucmVxdWlyZXNfZ3JhZCkpOgogICAgICAgIHJldHVybiAiZGlyZWN0X2F1dG9ncmFkX3VzZSIKICAgIHByb3A9dG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoQS5kZXZpY2UpCiAgICBpZiAocHJvcC5tYWpvcixwcm9wLm1pbm9yLHByb3AubXVsdGlfcHJvY2Vzc29yX2NvdW50KSAhPSAoNyw1LDQwKToKICAgICAgICByZXR1cm4gImFyY2hpdGVjdHVyZSIKICAgIHJldHVybiBOb25lCgoKZGVmIG1ha2VfcmVwbGFjZW1lbnQobmF0aXZlKToKICAgIGRlZiByZXBsYWNlbWVudChBLEIsc2hhcGVCLGFic21heCxibG9ja3NpemUscXVhbnRfdHlwZSxiaWFzPU5vbmUsCiAgICAgICAgICAgICAgICAgICAgYWJzbWF4XzhiaXQ9Tm9uZSxhYnNtYXhfY29kZT1Ob25lLGFic21heF9vZmZzZXQ9Tm9uZSk6CiAgICAgICAgYXJncz0oQSxCLHNoYXBlQixhYnNtYXgsYmxvY2tzaXplLHF1YW50X3R5cGUsYmlhcyxhYnNtYXhfOGJpdCxhYnNtYXhfY29kZSxhYnNtYXhfb2Zmc2V0KQogICAgICAgIHJhd19iaWFzLHdoeT1ub3JtYWxpemVkX25hdGl2ZV9iaWFzKGJpYXMpCiAgICAgICAgaWYgd2h5IGlzIE5vbmU6CiAgICAgICAgICAgIG5vcm1hbGl6ZWQ9KEEsQixzaGFwZUIsYWJzbWF4LGJsb2Nrc2l6ZSxxdWFudF90eXBlLHJhd19iaWFzLGFic21heF84Yml0LGFic21heF9jb2RlLGFic21heF9vZmZzZXQpCiAgICAgICAgICAgIHdoeT1yZWFzb24oKm5vcm1hbGl6ZWQpCiAgICAgICAgaWYgd2h5IGlzIG5vdCBOb25lOgogICAgICAgICAgICBDT1VOVFNbIm5hdGl2ZSJdKz0xCiAgICAgICAgICAgIEZBTExCQUNLX1JFQVNPTlNbd2h5XT1GQUxMQkFDS19SRUFTT05TLmdldCh3aHksMCkrMQogICAgICAgICAgICByZXR1cm4gbmF0aXZlKCphcmdzKQogICAgICAgIG4saz1zaGFwZUI7bT1BLm51bWVsKCkvL2sKICAgICAgICBvdXQ9dG9yY2guZW1wdHkoKCpBLnNoYXBlWzotMV0sbiksZGV2aWNlPUEuZGV2aWNlLGR0eXBlPXRvcmNoLmZsb2F0MTYpCiAgICAgICAgY29tcGlsZWQ9c291cF9uZjRfbW1hX3RpbGVkX3JldXNlWyh0cml0b24uY2RpdihtLDY0KSx0cml0b24uY2RpdihuLDY0KSldKAogICAgICAgICAgICBBLEIsYWJzbWF4XzhiaXQsYWJzbWF4X2NvZGUsYWJzbWF4LGFic21heF9vZmZzZXQsQSBpZiByYXdfYmlhcyBpcyBOb25lIGVsc2UgcmF3X2JpYXMsb3V0LAogICAgICAgICAgICBtLG4sayxiaWFzIGlzIG5vdCBOb25lLG51bV93YXJwcz00LG51bV9zdGFnZXM9MSxlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgICAgIENPTVBJTEVEWyhuLGssYmlhcyBpcyBub3QgTm9uZSldPWNvbXBpbGVkCiAgICAgICAgQ09VTlRTWyJjdXN0b20iXSs9MQogICAgICAgIHJldHVybiBvdXQKICAgIHJldHVybiByZXBsYWNlbWVudAoKCmRlZiB2ZXJpZnlfbmF0aXZlX293bmVyKGtleSx2YWx1ZSxleHBlY3RlZCk6CiAgICBvd25lcnM9eydmYWxsYmFjayc6KCdiaXRzYW5kYnl0ZXMuYmFja2VuZHMuY3VkYS5vcHMnLCdfZGVxdWFudF9saW5lYXJfZmFsbGJhY2snKSwKICAgICAgICAgICAgJ2JhY2t3YXJkJzooJ2JpdHNhbmRieXRlcy5hdXRvZ3JhZC5fZnVuY3Rpb25zJywnTWF0TXVsNEJpdC5iYWNrd2FyZCcpfQogICAgaWYgKG5vdCBpbnNwZWN0LmlzZnVuY3Rpb24odmFsdWUpIG9yIGhhc2F0dHIodmFsdWUsJ19fd3JhcHBlZF9fJykKICAgICAgICAgICAgb3IgKHZhbHVlLl9fbW9kdWxlX18sdmFsdWUuX19xdWFsbmFtZV9fKSE9b3duZXJzW2tleV0pOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRm9yZWlnbiBuYXRpdmUgY2FsbGFibGUgb3duZXI6ICcra2V5KQogICAgaWYgaGFzaGxpYi5zaGEyNTYoaW5zcGVjdC5nZXRzb3VyY2UodmFsdWUpLmVuY29kZSgpKS5oZXhkaWdlc3QoKSE9ZXhwZWN0ZWRba2V5XToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBvd25lcnNoaXAgc291cmNlIG1pc21hdGNoOiAnK2tleSkKCgpAY29udGV4dG1hbmFnZXIKZGVmIHRlbXBvcmFyeV9uYXRpdmVfZmFsbGJhY2soZXhwZWN0ZWQpOgogICAgIiIiT25seSBmb3IgY29udHJvbGxlZCBtaWNyb2JlbmNoIGNoaWxkcmVuOyBmYWlsIGNsb3NlZCBvbiBmb3JlaWduIG93bmVycy4iIiIKICAgIGltcG9ydCBiaXRzYW5kYnl0ZXMuYmFja2VuZHMuY3VkYS5vcHMgYXMgb3BzCiAgICBmcm9tIGJpdHNhbmRieXRlcy5hdXRvZ3JhZC5fZnVuY3Rpb25zIGltcG9ydCBNYXRNdWw0Qml0CiAgICBuYXRpdmU9b3BzLl9kZXF1YW50X2xpbmVhcl9mYWxsYmFjaztiYWNrd2FyZD1NYXRNdWw0Qml0LmJhY2t3YXJkCiAgICBmb3Iga2V5LHZhbHVlIGluICgoImZhbGxiYWNrIixuYXRpdmUpLCgiYmFja3dhcmQiLGJhY2t3YXJkKSk6CiAgICAgICAgdmVyaWZ5X25hdGl2ZV9vd25lcihrZXksdmFsdWUsZXhwZWN0ZWQpCiAgICByZXBsYWNlbWVudD1tYWtlX3JlcGxhY2VtZW50KG5hdGl2ZSkKICAgIG9wcy5fZGVxdWFudF9saW5lYXJfZmFsbGJhY2s9cmVwbGFjZW1lbnQKICAgIHRyeToKICAgICAgICB5aWVsZCByZXBsYWNlbWVudAogICAgICAgIGlmIE1hdE11bDRCaXQuYmFja3dhcmQgaXMgbm90IGJhY2t3YXJkOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk5hdGl2ZSBiYWNrd2FyZCBvd25lciBjaGFuZ2VkIikKICAgICAgICBpZiBvcHMuX2RlcXVhbnRfbGluZWFyX2ZhbGxiYWNrIGlzIG5vdCByZXBsYWNlbWVudDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJGYWxsYmFjayBvd25lciBjaGFuZ2VkIikKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgb3BzLl9kZXF1YW50X2xpbmVhcl9mYWxsYmFjayBpcyByZXBsYWNlbWVudDoKICAgICAgICAgICAgb3BzLl9kZXF1YW50X2xpbmVhcl9mYWxsYmFjaz1uYXRpdmUK', 'solution/kernel.py': 'IiIiT3JpZ2luYWw2NHg2NC80LXdhcnAgTkY0IE1NQSB3aXRoMzJ4MzIgcGVyLXdhcnAgcmVnaXN0ZXIgb3BlcmFuZCByZXVzZS4KClNhbWUgcHVibGljIG0xNm44azggaW5zdHJ1Y3Rpb24gbWFwcGluZyBhbmQgc2NhbGFyIGFyaXRobWV0aWMgYXMgb3JpZ2luYWwgdjQ7CkEgZnJhZ21lbnRzIHNoYXJlZCBieTQgTi1mcmFnbWVudHMsIGRlY29kZWQgQiBieTIgTS1mcmFnbWVudHMuIE5vIHZlbmRvciBjb2RlLgpSZXF1aXJlcyBvbmUgbG9naWNhbCBlbGVtZW50IHBlciB0aHJlYWQsNCBjb250aWd1b3VzIHBoeXNpY2FsIHdhcnBzLCBhbmQgQTQtYnl0ZSBhbGlnbm1lbnQuCiIiIgppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKZnJvbSAubmY0X2Zvcm1hdCBpbXBvcnQgX25mNF92YWx1ZQoKCkB0cml0b24uaml0CmRlZiBfcGFja19oYWxmMihhLGIpOgogICAgcmV0dXJuIGEudG8odGwudWludDE2LGJpdGNhc3Q9VHJ1ZSkudG8odGwudWludDMyKSB8IChiLnRvKHRsLnVpbnQxNixiaXRjYXN0PVRydWUpLnRvKHRsLnVpbnQzMik8PDE2KQoKCkB0cml0b24uaml0CmRlZiBfbW1hMTY4OChhMCxhMSxiMCxjMCxjMSxjMixjMyk6CiAgICByZXR1cm4gdGwuaW5saW5lX2FzbV9lbGVtZW50d2lzZSgKICAgICAgICAibW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyIHskMCwkMSwkMiwkM30sIHskNCwkNX0sIHskNn0sIHskNywkOCwkOSwkMTB9OyIsCiAgICAgICAgY29uc3RyYWludHM9Ij1mLD1mLD1mLD1mLHIscixyLGYsZixmLGYiLAogICAgICAgIGFyZ3M9W2EwLGExLGIwLGMwLGMxLGMyLGMzXSxkdHlwZT0odGwuZmxvYXQzMix0bC5mbG9hdDMyLHRsLmZsb2F0MzIsdGwuZmxvYXQzMiksCiAgICAgICAgaXNfcHVyZT1UcnVlLHBhY2s9MSkKCgpAdHJpdG9uLmppdChkb19ub3Rfc3BlY2lhbGl6ZT1bJ00nXSkKZGVmIHNvdXBfbmY0X21tYV90aWxlZF9yZXVzZShBLFBBQ0tFRCxRQUJTLENPREUyLEFCUzIsT0ZGU0VULEJpYXMsT3V0LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIE0sTjp0bC5jb25zdGV4cHIsSzp0bC5jb25zdGV4cHIsQklBUzp0bC5jb25zdGV4cHIpOgogICAgdGwuc3RhdGljX2Fzc2VydChLJTY0PT0wKQogICAgdGhyZWFkPXRsLmFyYW5nZSgwLDEyOCkKICAgIHdhcnA9dGhyZWFkLy8zMjtsYW5lPXRocmVhZCUzMgogICAgZ3JvdXA9bGFuZS8vNDtwYWlyPWxhbmUlNAogICAgbTA9dGwucHJvZ3JhbV9pZCgwKSo2NCsod2FycC8vMikqMzIrZ3JvdXAKICAgIG4wPXRsLnByb2dyYW1faWQoMSkqNjQrKHdhcnAlMikqMzIrZ3JvdXAKICAgIEEyPXRsLmNhc3QoQSx0bC5wb2ludGVyX3R5cGUodGwudWludDMyKSkKICAgIGMwMF8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzAwXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMDBfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MwMF8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMwMV8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzAxXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMDFfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MwMV8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMwMl8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzAyXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMDJfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MwMl8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMwM18wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzAzXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMDNfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MwM18zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMxMF8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzEwXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMTBfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MxMF8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMxMV8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzExXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMTFfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MxMV8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMxMl8wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzEyXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMTJfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MxMl8zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGMxM18wPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMik7YzEzXzE9dGwuZnVsbCgoMTI4LCksMCx0bC5mbG9hdDMyKTtjMTNfMj10bC5mdWxsKCgxMjgsKSwwLHRsLmZsb2F0MzIpO2MxM18zPXRsLmZ1bGwoKDEyOCwpLDAsdGwuZmxvYXQzMikKICAgIGZvciBibG9jayBpbiB0bC5yYW5nZSgwLEsvLzY0LGxvb3BfdW5yb2xsX2ZhY3Rvcj0xKToKICAgICAgICBuczA9bjArMAogICAgICAgIHNiMD1uczAqKEsvLzY0KStibG9jawogICAgICAgIHFpMD10bC5sb2FkKFFBQlMrc2IwLG5zMDxOLG90aGVyPTApLnRvKHRsLmludDMyKQogICAgICAgIGN2MD10bC5sb2FkKENPREUyK3FpMCxuczA8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIGFiMD10bC5sb2FkKEFCUzIrc2IwLy8yNTYsbnMwPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBwcm9kdWN0MD1jdjAqYWIwCiAgICAgICAgc2NhbGUwPXByb2R1Y3QwK3RsLmxvYWQoT0ZGU0VUKS50byh0bC5mbG9hdDMyKQogICAgICAgIG5zMT1uMCs4CiAgICAgICAgc2IxPW5zMSooSy8vNjQpK2Jsb2NrCiAgICAgICAgcWkxPXRsLmxvYWQoUUFCUytzYjEsbnMxPE4sb3RoZXI9MCkudG8odGwuaW50MzIpCiAgICAgICAgY3YxPXRsLmxvYWQoQ09ERTIrcWkxLG5zMTxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYWIxPXRsLmxvYWQoQUJTMitzYjEvLzI1NixuczE8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIHByb2R1Y3QxPWN2MSphYjEKICAgICAgICBzY2FsZTE9cHJvZHVjdDErdGwubG9hZChPRkZTRVQpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgbnMyPW4wKzE2CiAgICAgICAgc2IyPW5zMiooSy8vNjQpK2Jsb2NrCiAgICAgICAgcWkyPXRsLmxvYWQoUUFCUytzYjIsbnMyPE4sb3RoZXI9MCkudG8odGwuaW50MzIpCiAgICAgICAgY3YyPXRsLmxvYWQoQ09ERTIrcWkyLG5zMjxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYWIyPXRsLmxvYWQoQUJTMitzYjIvLzI1NixuczI8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIHByb2R1Y3QyPWN2MiphYjIKICAgICAgICBzY2FsZTI9cHJvZHVjdDIrdGwubG9hZChPRkZTRVQpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgbnMzPW4wKzI0CiAgICAgICAgc2IzPW5zMyooSy8vNjQpK2Jsb2NrCiAgICAgICAgcWkzPXRsLmxvYWQoUUFCUytzYjMsbnMzPE4sb3RoZXI9MCkudG8odGwuaW50MzIpCiAgICAgICAgY3YzPXRsLmxvYWQoQ09ERTIrcWkzLG5zMzxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYWIzPXRsLmxvYWQoQUJTMitzYjMvLzI1NixuczM8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIHByb2R1Y3QzPWN2MyphYjMKICAgICAgICBzY2FsZTM9cHJvZHVjdDMrdGwubG9hZChPRkZTRVQpLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgZm9yIGNodW5rIGluIHRsLnJhbmdlKDAsOCxsb29wX3Vucm9sbF9mYWN0b3I9MSk6CiAgICAgICAgICAgIGswPWJsb2NrKjY0K2NodW5rKjgrcGFpcioyCiAgICAgICAgICAgIGEwMD10bC5sb2FkKEEyKygobTArMCkqSytrMCkvLzIsbTArMDxNLG90aGVyPTApCiAgICAgICAgICAgIGEwMT10bC5sb2FkKEEyKygobTArOCkqSytrMCkvLzIsbTArODxNLG90aGVyPTApCiAgICAgICAgICAgIGExMD10bC5sb2FkKEEyKygobTArMTYpKksrazApLy8yLG0wKzE2PE0sb3RoZXI9MCkKICAgICAgICAgICAgYTExPXRsLmxvYWQoQTIrKChtMCsyNCkqSytrMCkvLzIsbTArMjQ8TSxvdGhlcj0wKQogICAgICAgICAgICBieXRlMD10bC5sb2FkKFBBQ0tFRCsobnMwKksrazApLy8yLG5zMDxOLG90aGVyPTApLnRvKHRsLnVpbnQzMikKICAgICAgICAgICAgYmgwPShfbmY0X3ZhbHVlKGJ5dGUwPj40KSpzY2FsZTApLnRvKHRsLmZsb2F0MTYsZnBfZG93bmNhc3Rfcm91bmRpbmc9J3J0bmUnKQogICAgICAgICAgICBibDA9KF9uZjRfdmFsdWUoYnl0ZTAmMTUpKnNjYWxlMCkudG8odGwuZmxvYXQxNixmcF9kb3duY2FzdF9yb3VuZGluZz0ncnRuZScpCiAgICAgICAgICAgIGIwPV9wYWNrX2hhbGYyKGJoMCxibDApCiAgICAgICAgICAgIGJ5dGUxPXRsLmxvYWQoUEFDS0VEKyhuczEqSytrMCkvLzIsbnMxPE4sb3RoZXI9MCkudG8odGwudWludDMyKQogICAgICAgICAgICBiaDE9KF9uZjRfdmFsdWUoYnl0ZTE+PjQpKnNjYWxlMSkudG8odGwuZmxvYXQxNixmcF9kb3duY2FzdF9yb3VuZGluZz0ncnRuZScpCiAgICAgICAgICAgIGJsMT0oX25mNF92YWx1ZShieXRlMSYxNSkqc2NhbGUxKS50byh0bC5mbG9hdDE2LGZwX2Rvd25jYXN0X3JvdW5kaW5nPSdydG5lJykKICAgICAgICAgICAgYjE9X3BhY2tfaGFsZjIoYmgxLGJsMSkKICAgICAgICAgICAgYnl0ZTI9dGwubG9hZChQQUNLRUQrKG5zMipLK2swKS8vMixuczI8TixvdGhlcj0wKS50byh0bC51aW50MzIpCiAgICAgICAgICAgIGJoMj0oX25mNF92YWx1ZShieXRlMj4+NCkqc2NhbGUyKS50byh0bC5mbG9hdDE2LGZwX2Rvd25jYXN0X3JvdW5kaW5nPSdydG5lJykKICAgICAgICAgICAgYmwyPShfbmY0X3ZhbHVlKGJ5dGUyJjE1KSpzY2FsZTIpLnRvKHRsLmZsb2F0MTYsZnBfZG93bmNhc3Rfcm91bmRpbmc9J3J0bmUnKQogICAgICAgICAgICBiMj1fcGFja19oYWxmMihiaDIsYmwyKQogICAgICAgICAgICBieXRlMz10bC5sb2FkKFBBQ0tFRCsobnMzKksrazApLy8yLG5zMzxOLG90aGVyPTApLnRvKHRsLnVpbnQzMikKICAgICAgICAgICAgYmgzPShfbmY0X3ZhbHVlKGJ5dGUzPj40KSpzY2FsZTMpLnRvKHRsLmZsb2F0MTYsZnBfZG93bmNhc3Rfcm91bmRpbmc9J3J0bmUnKQogICAgICAgICAgICBibDM9KF9uZjRfdmFsdWUoYnl0ZTMmMTUpKnNjYWxlMykudG8odGwuZmxvYXQxNixmcF9kb3duY2FzdF9yb3VuZGluZz0ncnRuZScpCiAgICAgICAgICAgIGIzPV9wYWNrX2hhbGYyKGJoMyxibDMpCiAgICAgICAgICAgIGMwMF8wLGMwMF8xLGMwMF8yLGMwMF8zPV9tbWExNjg4KGEwMCxhMDEsYjAsYzAwXzAsYzAwXzEsYzAwXzIsYzAwXzMpCiAgICAgICAgICAgIGMwMV8wLGMwMV8xLGMwMV8yLGMwMV8zPV9tbWExNjg4KGEwMCxhMDEsYjEsYzAxXzAsYzAxXzEsYzAxXzIsYzAxXzMpCiAgICAgICAgICAgIGMwMl8wLGMwMl8xLGMwMl8yLGMwMl8zPV9tbWExNjg4KGEwMCxhMDEsYjIsYzAyXzAsYzAyXzEsYzAyXzIsYzAyXzMpCiAgICAgICAgICAgIGMwM18wLGMwM18xLGMwM18yLGMwM18zPV9tbWExNjg4KGEwMCxhMDEsYjMsYzAzXzAsYzAzXzEsYzAzXzIsYzAzXzMpCiAgICAgICAgICAgIGMxMF8wLGMxMF8xLGMxMF8yLGMxMF8zPV9tbWExNjg4KGExMCxhMTEsYjAsYzEwXzAsYzEwXzEsYzEwXzIsYzEwXzMpCiAgICAgICAgICAgIGMxMV8wLGMxMV8xLGMxMV8yLGMxMV8zPV9tbWExNjg4KGExMCxhMTEsYjEsYzExXzAsYzExXzEsYzExXzIsYzExXzMpCiAgICAgICAgICAgIGMxMl8wLGMxMl8xLGMxMl8yLGMxMl8zPV9tbWExNjg4KGExMCxhMTEsYjIsYzEyXzAsYzEyXzEsYzEyXzIsYzEyXzMpCiAgICAgICAgICAgIGMxM18wLGMxM18xLGMxM18yLGMxM18zPV9tbWExNjg4KGExMCxhMTEsYjMsYzEzXzAsYzEzXzEsYzEzXzIsYzEzXzMpCiAgICBjb2wwPXRsLnByb2dyYW1faWQoMSkqNjQrKHdhcnAlMikqMzIrcGFpcioyCiAgICBjb2wwPWNvbDArMAogICAgaWYgQklBUzoKICAgICAgICBiaWFzMDA9dGwubG9hZChCaWFzK2NvbDAsY29sMDxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYmlhczAxPXRsLmxvYWQoQmlhcytjb2wwKzEsY29sMCsxPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBjMDBfMD1jMDBfMCtiaWFzMDAKICAgICAgICBjMDBfMT1jMDBfMStiaWFzMDEKICAgICAgICBjMDBfMj1jMDBfMitiaWFzMDAKICAgICAgICBjMDBfMz1jMDBfMytiaWFzMDEKICAgICAgICBjMTBfMD1jMTBfMCtiaWFzMDAKICAgICAgICBjMTBfMT1jMTBfMStiaWFzMDEKICAgICAgICBjMTBfMj1jMTBfMitiaWFzMDAKICAgICAgICBjMTBfMz1jMTBfMytiaWFzMDEKICAgIGNvbDE9Y29sMCs4CiAgICBpZiBCSUFTOgogICAgICAgIGJpYXMxMD10bC5sb2FkKEJpYXMrY29sMSxjb2wxPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBiaWFzMTE9dGwubG9hZChCaWFzK2NvbDErMSxjb2wxKzE8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIGMwMV8wPWMwMV8wK2JpYXMxMAogICAgICAgIGMwMV8xPWMwMV8xK2JpYXMxMQogICAgICAgIGMwMV8yPWMwMV8yK2JpYXMxMAogICAgICAgIGMwMV8zPWMwMV8zK2JpYXMxMQogICAgICAgIGMxMV8wPWMxMV8wK2JpYXMxMAogICAgICAgIGMxMV8xPWMxMV8xK2JpYXMxMQogICAgICAgIGMxMV8yPWMxMV8yK2JpYXMxMAogICAgICAgIGMxMV8zPWMxMV8zK2JpYXMxMQogICAgY29sMj1jb2wwKzE2CiAgICBpZiBCSUFTOgogICAgICAgIGJpYXMyMD10bC5sb2FkKEJpYXMrY29sMixjb2wyPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBiaWFzMjE9dGwubG9hZChCaWFzK2NvbDIrMSxjb2wyKzE8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIGMwMl8wPWMwMl8wK2JpYXMyMAogICAgICAgIGMwMl8xPWMwMl8xK2JpYXMyMQogICAgICAgIGMwMl8yPWMwMl8yK2JpYXMyMAogICAgICAgIGMwMl8zPWMwMl8zK2JpYXMyMQogICAgICAgIGMxMl8wPWMxMl8wK2JpYXMyMAogICAgICAgIGMxMl8xPWMxMl8xK2JpYXMyMQogICAgICAgIGMxMl8yPWMxMl8yK2JpYXMyMAogICAgICAgIGMxMl8zPWMxMl8zK2JpYXMyMQogICAgY29sMz1jb2wwKzI0CiAgICBpZiBCSUFTOgogICAgICAgIGJpYXMzMD10bC5sb2FkKEJpYXMrY29sMyxjb2wzPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBiaWFzMzE9dGwubG9hZChCaWFzK2NvbDMrMSxjb2wzKzE8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIGMwM18wPWMwM18wK2JpYXMzMAogICAgICAgIGMwM18xPWMwM18xK2JpYXMzMQogICAgICAgIGMwM18yPWMwM18yK2JpYXMzMAogICAgICAgIGMwM18zPWMwM18zK2JpYXMzMQogICAgICAgIGMxM18wPWMxM18wK2JpYXMzMAogICAgICAgIGMxM18xPWMxM18xK2JpYXMzMQogICAgICAgIGMxM18yPWMxM18yK2JpYXMzMAogICAgICAgIGMxM18zPWMxM18zK2JpYXMzMQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDArMCxjMDBfMC50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMCswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDArMSxjMDBfMS50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMCsxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDArMCxjMDBfMi50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMCswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDArMSxjMDBfMy50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMCsxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDErMCxjMDFfMC50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMSswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDErMSxjMDFfMS50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMSsxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDErMCxjMDFfMi50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMSswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDErMSxjMDFfMy50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMSsxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDIrMCxjMDJfMC50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMiswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDIrMSxjMDJfMS50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMisxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDIrMCxjMDJfMi50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMiswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDIrMSxjMDJfMy50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMisxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDMrMCxjMDNfMC50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMyswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCswKSpOK2NvbDMrMSxjMDNfMS50byh0bC5mbG9hdDE2KSwobTArMDxNKSYoY29sMysxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDMrMCxjMDNfMi50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMyswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCs4KSpOK2NvbDMrMSxjMDNfMy50byh0bC5mbG9hdDE2KSwobTArODxNKSYoY29sMysxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsxNikqTitjb2wwKzAsYzEwXzAudG8odGwuZmxvYXQxNiksKG0wKzE2PE0pJihjb2wwKzA8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzE2KSpOK2NvbDArMSxjMTBfMS50byh0bC5mbG9hdDE2KSwobTArMTY8TSkmKGNvbDArMTxOKSkKICAgIHRsLnN0b3JlKE91dCsobTArMjQpKk4rY29sMCswLGMxMF8yLnRvKHRsLmZsb2F0MTYpLChtMCsyNDxNKSYoY29sMCswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsyNCkqTitjb2wwKzEsYzEwXzMudG8odGwuZmxvYXQxNiksKG0wKzI0PE0pJihjb2wwKzE8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzE2KSpOK2NvbDErMCxjMTFfMC50byh0bC5mbG9hdDE2KSwobTArMTY8TSkmKGNvbDErMDxOKSkKICAgIHRsLnN0b3JlKE91dCsobTArMTYpKk4rY29sMSsxLGMxMV8xLnRvKHRsLmZsb2F0MTYpLChtMCsxNjxNKSYoY29sMSsxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsyNCkqTitjb2wxKzAsYzExXzIudG8odGwuZmxvYXQxNiksKG0wKzI0PE0pJihjb2wxKzA8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzI0KSpOK2NvbDErMSxjMTFfMy50byh0bC5mbG9hdDE2KSwobTArMjQ8TSkmKGNvbDErMTxOKSkKICAgIHRsLnN0b3JlKE91dCsobTArMTYpKk4rY29sMiswLGMxMl8wLnRvKHRsLmZsb2F0MTYpLChtMCsxNjxNKSYoY29sMiswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsxNikqTitjb2wyKzEsYzEyXzEudG8odGwuZmxvYXQxNiksKG0wKzE2PE0pJihjb2wyKzE8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzI0KSpOK2NvbDIrMCxjMTJfMi50byh0bC5mbG9hdDE2KSwobTArMjQ8TSkmKGNvbDIrMDxOKSkKICAgIHRsLnN0b3JlKE91dCsobTArMjQpKk4rY29sMisxLGMxMl8zLnRvKHRsLmZsb2F0MTYpLChtMCsyNDxNKSYoY29sMisxPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsxNikqTitjb2wzKzAsYzEzXzAudG8odGwuZmxvYXQxNiksKG0wKzE2PE0pJihjb2wzKzA8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzE2KSpOK2NvbDMrMSxjMTNfMS50byh0bC5mbG9hdDE2KSwobTArMTY8TSkmKGNvbDMrMTxOKSkKICAgIHRsLnN0b3JlKE91dCsobTArMjQpKk4rY29sMyswLGMxM18yLnRvKHRsLmZsb2F0MTYpLChtMCsyNDxNKSYoY29sMyswPE4pKQogICAgdGwuc3RvcmUoT3V0KyhtMCsyNCkqTitjb2wzKzEsYzEzXzMudG8odGwuZmxvYXQxNiksKG0wKzI0PE0pJihjb2wzKzE8TikpCg==', 'solution/module_preflight.py': 'IiIiUmVhbCBuYXRpdmUgTGluZWFyNGJpdC9QYXJhbWV0ZXIgYWN0aXZhdGlvbjsgc3ludGhldGljIHN0YW5kYWxvbmUgaW5wdXRzIG9ubHkuCgpUaGlzIHRhcmdldC1vbmx5IGdhdGUgZm9sbG93cyB0aGUgc291cmNlLWJvdW5kIHJldmlzZWQgTU1BIGFyaXRobWV0aWMgZ2F0ZSBhbmQKcnVucyBiZWZvcmUgYW55IHRpbWluZy4gTm8gcHJldHJhaW5lZCBtb2RlbCwgdGFzayBhY3RpdmF0aW9uIG9yIHRyYWluaW5nIGNsYWltLgoiIiIKaW1wb3J0IGFyZ3BhcnNlLGhhc2hsaWIsaW5zcGVjdCxqc29uLG9zLHRyYWNlYmFjawpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCgpQUk9HUkVTUz17fQoKZGVmIHJ1bigpOgogICAgaWYgb3MuZW52aXJvbi5nZXQoJ1RSSVRPTl9JTlRFUlBSRVQnKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1RhcmdldCBoYXJkd2FyZSByZXF1aXJlZCcpCiAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLGJpdHNhbmRieXRlcyBhcyBibmIKICAgIGZyb20gYml0c2FuZGJ5dGVzLm5uIGltcG9ydCBMaW5lYXI0Yml0LFBhcmFtczRiaXQKICAgIGZyb20gYml0c2FuZGJ5dGVzLmF1dG9ncmFkLl9mdW5jdGlvbnMgaW1wb3J0IE1hdE11bDRCaXQKICAgIGZyb20gLiBpbXBvcnQgZ3VhcmRlZF9mb3J3YXJkIGFzIGcKICAgIGZyb20gLmZpeHR1cmVzIGltcG9ydCBFWFBFQ1RFRCxleGFjdCxlbmdpbmVlcmluZ19jbG9zZQogICAgaWYgKHRvcmNoLl9fdmVyc2lvbl9fIT0nMi4xMS4wK2N1MTI4JyBvciB0cml0b24uX192ZXJzaW9uX18hPSczLjYuMCcgb3IgYm5iLl9fdmVyc2lvbl9fIT0nMC41MC4yJwogICAgICAgICAgICBvciB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpIT0xIG9yIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApIT0nVGVzbGEgVDQnCiAgICAgICAgICAgIG9yIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApIT0oNyw1KSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdFeGFjdCB0YXJnZXQgcnVudGltZSByZXF1aXJlZCcpCiAgICBmcm9tIC5ndWFyZGVkX2ZvcndhcmQgaW1wb3J0IHZlcmlmeV9uYXRpdmVfb3duZXIKICAgIG1vZHVsZV9mb3J3YXJkPUxpbmVhcjRiaXQuZm9yd2FyZDthdXRvZ3JhZF9mb3J3YXJkPU1hdE11bDRCaXQuZm9yd2FyZAogICAgZXhwZWN0ZWRfb3duZXJzPXsKICAgICAgICAnbGluZWFyNGJpdF9mb3J3YXJkJzooJ2JpdHNhbmRieXRlcy5ubi5tb2R1bGVzJywnTGluZWFyNGJpdC5mb3J3YXJkJywnZDUyY2Y2MzQ1NzcxN2NjNzE4Y2EwYTQ0OWRhZDA5NjcxNzFlNjc5YmU5MGQ1ZDJhZGNhZmUyZThhMTQxMDkzZCcpLAogICAgICAgICdtYXRtdWw0Yml0X2ZvcndhcmQnOignYml0c2FuZGJ5dGVzLmF1dG9ncmFkLl9mdW5jdGlvbnMnLCdNYXRNdWw0Qml0LmZvcndhcmQnLCcyOWY0ZTIxYTRmZTk5ZGIzZjZlMGY5NGRiZjZjYmU4NDI2MjIwZjkxNjNjMDUwMWM4NmQzM2YzNjJhNTRjY2I2JyksCiAgICB9CiAgICBmb3Iga2V5LGZuIGluICgoJ2xpbmVhcjRiaXRfZm9yd2FyZCcsbW9kdWxlX2ZvcndhcmQpLCgnbWF0bXVsNGJpdF9mb3J3YXJkJyxhdXRvZ3JhZF9mb3J3YXJkKSk6CiAgICAgICAgb3duZXIscXVhbG5hbWUsZGlnZXN0PWV4cGVjdGVkX293bmVyc1trZXldCiAgICAgICAgaWYgKG5vdCBpbnNwZWN0LmlzZnVuY3Rpb24oZm4pIG9yIGhhc2F0dHIoZm4sJ19fd3JhcHBlZF9fJykgb3IgKGZuLl9fbW9kdWxlX18sZm4uX19xdWFsbmFtZV9fKSE9KG93bmVyLHF1YWxuYW1lKQogICAgICAgICAgICAgICAgb3IgaGFzaGxpYi5zaGEyNTYoaW5zcGVjdC5nZXRzb3VyY2UoZm4pLmVuY29kZSgpKS5oZXhkaWdlc3QoKSE9ZGlnZXN0KToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdOYXRpdmUgbW9kdWxlIGZvcndhcmQgc291cmNlL293bmVyIG1pc21hdGNoOiAnK2tleSkKICAgIGJhY2t3YXJkPU1hdE11bDRCaXQuYmFja3dhcmQ7cm93cz1bXTtQUk9HUkVTU1sncm93cyddPXJvd3MKICAgIGNvdW50c19iZWZvcmU9ZGljdChnLkNPVU5UUykKICAgIGZvciBuIGluICgyNTYsMTUzNik6CiAgICAgICAgdG9yY2gubWFudWFsX3NlZWQoNzgrbikKICAgICAgICBtb2R1bGU9TGluZWFyNGJpdCgxNTM2LG4sYmlhcz1UcnVlLGNvbXB1dGVfZHR5cGU9dG9yY2guZmxvYXQxNixjb21wcmVzc19zdGF0aXN0aWNzPVRydWUscXVhbnRfdHlwZT0nbmY0JykKICAgICAgICBzb3VyY2U9dG9yY2gucmFuZG4obiwxNTM2LGR0eXBlPXRvcmNoLmZsb2F0MTYpKi4xCiAgICAgICAgbW9kdWxlLndlaWdodD1QYXJhbXM0Yml0KHNvdXJjZSxyZXF1aXJlc19ncmFkPUZhbHNlLGNvbXByZXNzX3N0YXRpc3RpY3M9VHJ1ZSxxdWFudF90eXBlPSduZjQnKQogICAgICAgIG1vZHVsZS5iaWFzPXRvcmNoLm5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbihuLGR0eXBlPXRvcmNoLmZsb2F0MTYpKi4wMyxyZXF1aXJlc19ncmFkPUZhbHNlKQogICAgICAgIG1vZHVsZT1tb2R1bGUudG8oJ2N1ZGEnKTttb2R1bGUudHJhaW4oKQogICAgICAgIGlmIHR5cGUobW9kdWxlLmJpYXMpIGlzIG5vdCB0b3JjaC5ubi5QYXJhbWV0ZXIgb3IgbW9kdWxlLmJpYXMuZHR5cGUhPXRvcmNoLmZsb2F0MTY6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQWN0dWFsIG5hdGl2ZSBQYXJhbWV0ZXIgZml4dHVyZSBub3QgZXN0YWJsaXNoZWQnKQogICAgICAgIHdlaWdodF9pZD1pZChtb2R1bGUud2VpZ2h0KTtiaWFzX2lkPWlkKG1vZHVsZS5iaWFzKQogICAgICAgIGJhc2U9dG9yY2gucmFuZG4oNCwxNywxNTM2LGRldmljZT0nY3VkYScsZHR5cGU9dG9yY2guZmxvYXQxNikqLjEKICAgICAgICBwYWNrZWRfYmVmb3JlPW1vZHVsZS53ZWlnaHQuZGV0YWNoKCkuY2xvbmUoKTtiaWFzX2JlZm9yZT1tb2R1bGUuYmlhcy5kZXRhY2goKS5jbG9uZSgpCiAgICAgICAgZm9yIHRyYWluYWJsZV9iaWFzIGluIChGYWxzZSxUcnVlKToKICAgICAgICAgICAgbW9kdWxlLmJpYXMucmVxdWlyZXNfZ3JhZF8odHJhaW5hYmxlX2JpYXMpCiAgICAgICAgICAgIGZvciBzY2FsZSBpbiAoMS4sMTI4Liw2NTUzNi4pOgogICAgICAgICAgICAgICAgUFJPR1JFU1NbJ2N1cnJlbnRfY2FzZSddPXsnTic6biwnYmlhc190cmFpbmFibGUnOnRyYWluYWJsZV9iaWFzLCdzY2FsZSc6c2NhbGV9CiAgICAgICAgICAgICAgICBkZWYgb25lKGN1c3RvbSk6CiAgICAgICAgICAgICAgICAgICAgYT1iYXNlLmRldGFjaCgpLmNsb25lKCkucmVxdWlyZXNfZ3JhZF8oKQogICAgICAgICAgICAgICAgICAgIGJlZm9yZT10b3JjaC5jdWRhLmdldF9ybmdfc3RhdGUoKS5jbG9uZSgpCiAgICAgICAgICAgICAgICAgICAgaWYgY3VzdG9tOgogICAgICAgICAgICAgICAgICAgICAgICB3aXRoIGcudGVtcG9yYXJ5X25hdGl2ZV9mYWxsYmFjayhFWFBFQ1RFRCk6b3V0cHV0PW1vZHVsZShhKQogICAgICAgICAgICAgICAgICAgIGVsc2U6b3V0cHV0PW1vZHVsZShhKQogICAgICAgICAgICAgICAgICAgIGNvdGFuZ2VudD10b3JjaC5saW5zcGFjZSgtLjAxLC4wMSxvdXRwdXQubnVtZWwoKSxkZXZpY2U9J2N1ZGEnKS5yZXNoYXBlX2FzKG91dHB1dCkuaGFsZigpKnNjYWxlCiAgICAgICAgICAgICAgICAgICAgcGFyYW1ldGVycz0oYSxtb2R1bGUuYmlhcykgaWYgdHJhaW5hYmxlX2JpYXMgZWxzZSAoYSwpCiAgICAgICAgICAgICAgICAgICAgZ3JhZGllbnRzPXRvcmNoLmF1dG9ncmFkLmdyYWQob3V0cHV0LHBhcmFtZXRlcnMsY290YW5nZW50KQogICAgICAgICAgICAgICAgICAgIGlmIG5vdCB0b3JjaC5lcXVhbChiZWZvcmUsdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlKCkpOnJhaXNlIFJ1bnRpbWVFcnJvcignUk5HIGNoYW5nZWQnKQogICAgICAgICAgICAgICAgICAgIHJldHVybiBvdXRwdXQuZGV0YWNoKCksZ3JhZGllbnRzCiAgICAgICAgICAgICAgICBleHBlY3RlZCxyZWZlcmVuY2VfZ3JhZD1vbmUoRmFsc2UpO2NvdW50PWcuQ09VTlRTWydjdXN0b20nXTthY3R1YWwsYWN0dWFsX2dyYWQ9b25lKFRydWUpCiAgICAgICAgICAgICAgICBpZiBnLkNPVU5UU1snY3VzdG9tJ10hPWNvdW50KzE6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdBY3R1YWwgTGluZWFyNGJpdCBtb2R1bGUgaW5hY3RpdmU6ICcrcmVwcihnLkZBTExCQUNLX1JFQVNPTlMpKQogICAgICAgICAgICAgICAgZXJyb3I9ZW5naW5lZXJpbmdfY2xvc2UoYWN0dWFsLGV4cGVjdGVkKQogICAgICAgICAgICAgICAgZm9yIHgseSBpbiB6aXAoYWN0dWFsX2dyYWQscmVmZXJlbmNlX2dyYWQpOmV4YWN0KHgseSkKICAgICAgICAgICAgICAgIGlmIChpZChtb2R1bGUud2VpZ2h0KSE9d2VpZ2h0X2lkIG9yIGlkKG1vZHVsZS5iaWFzKSE9Ymlhc19pZAogICAgICAgICAgICAgICAgICAgICAgICBvciBNYXRNdWw0Qml0LmJhY2t3YXJkIGlzIG5vdCBiYWNrd2FyZCBvciB0eXBlKG1vZHVsZS5iaWFzKSBpcyBub3QgdG9yY2gubm4uUGFyYW1ldGVyCiAgICAgICAgICAgICAgICAgICAgICAgIG9yIExpbmVhcjRiaXQuZm9yd2FyZCBpcyBub3QgbW9kdWxlX2ZvcndhcmQgb3IgTWF0TXVsNEJpdC5mb3J3YXJkIGlzIG5vdCBhdXRvZ3JhZF9mb3J3YXJkKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ05hdGl2ZSBQYXJhbWV0ZXIvYmFja3dhcmQgaWRlbnRpdHkgY2hhbmdlZCcpCiAgICAgICAgICAgICAgICBleGFjdChtb2R1bGUud2VpZ2h0LmRldGFjaCgpLHBhY2tlZF9iZWZvcmUpO2V4YWN0KG1vZHVsZS5iaWFzLmRldGFjaCgpLGJpYXNfYmVmb3JlKQogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoeydOJzpuLCdLJzoxNTM2LCdzaGFwZSc6WzQsMTcsMTUzNl0sJ2JpYXNfdHJhaW5hYmxlJzp0cmFpbmFibGVfYmlhcywnc2NhbGUnOnNjYWxlLAogICAgICAgICAgICAgICAgICAgICdhY3R1YWxfbmF0aXZlX0xpbmVhcjRiaXRfZm9yd2FyZF9hY3RpdmUnOlRydWUsJ25hdGl2ZV9iYWNrd2FyZF9ncmFkaWVudHNfZXhhY3QnOlRydWUsCiAgICAgICAgICAgICAgICAgICAgJ2lucHV0X2tpbmQnOidzeW50aGV0aWMgc3RhbmRhbG9uZSBtb2R1bGUsIG5vdCBjYXB0dXJlZCBmaXhlZC10YXNrIHEvdiBhY3RpdmF0aW9ucycsCiAgICAgICAgICAgICAgICAgICAgJ3dlaWdodF9hbmRfYmlhc191bmNoYW5nZWQnOlRydWUsKiplcnJvcn0pCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIGlmIGxlbihyb3dzKSE9MTIgb3IgZy5DT1VOVFNbJ2N1c3RvbSddLWNvdW50c19iZWZvcmVbJ2N1c3RvbSddIT0xMjpyYWlzZSBSdW50aW1lRXJyb3IoJ0luY29tcGxldGUgYWN0dWFsIG1vZHVsZSBhY3RpdmF0aW9uIGdhdGUnKQogICAgcmV0dXJuIHsncm93cyc6cm93cywnY291bnRzJzpkaWN0KGcuQ09VTlRTKSwnY291bnRzX2JlZm9yZSc6Y291bnRzX2JlZm9yZSwnc291cmNlX293bmVycyc6ZXhwZWN0ZWRfb3duZXJzLAogICAgICAgICAgICAnZmFsbGJhY2tfcmVhc29ucyc6ZGljdChnLkZBTExCQUNLX1JFQVNPTlMpLAogICAgICAgICAgICAnZml4ZWRfdGFza190cmFpbmluZyc6RmFsc2UsJ3Rhc2tfcXVhbGl0eSc6J1VOS05PV04nfQoKCmRlZiBtYWluKCk6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tb3V0JyxyZXF1aXJlZD1UcnVlKTthcmdzPXAucGFyc2VfYXJncygpCiAgICByZXBvcnQ9eydzdGF0dXMnOidGQUlMRURfTU9EVUxFX1BSRUZMSUdIVCcsJ3Njb3BlJzonTUlDUk9CRU5DSF9PTkxZJywnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogICAgdHJ5OnJlcG9ydFsnY2hlY2tzJ109cnVuKCk7cmVwb3J0WydzdGF0dXMnXT0nUEFTU0VEX01PRFVMRV9QUkVGTElHSFRfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGV4YzpyZXBvcnQudXBkYXRlKGVycm9yPXJlcHIoZXhjKSx0cmFjZWJhY2s9dHJhY2ViYWNrLmZvcm1hdF9leGMoKSxwcm9ncmVzcz1QUk9HUkVTUykKICAgIFBhdGgoYXJncy5vdXQpLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIsYWxsb3dfbmFuPUZhbHNlKSsnXG4nKQogICAgcmFpc2UgU3lzdGVtRXhpdChyZXBvcnRbJ3N0YXR1cyddIT0nUEFTU0VEX01PRFVMRV9QUkVGTElHSFRfT05MWScpCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'solution/native_source_manifest.json': 'ewogICJmYWxsYmFjayI6ICJhZDA5Y2U4MWIyNWNiMGI4OTEzZjYxYzQ3MGIxYjc5NDJhYzQ1NGJmYzhhYTI3ZmUzYTVhYTdmYTBkMTVjYzZmIiwKICAiYmFja3dhcmQiOiAiMDUzZTg3MTkzMTlhNTI2YzYxZjI4OGEzNzIyMWJhNTIwMGRlOWVhYTIwZGFlN2M3ZTQ1Mzg2MzE1ZjI4NGNkMiIKfQo=', 'solution/nf4_format.py': 'IiIiUHVibGljIE5GNCBGUDMyIGZvcm1hdCBjb25zdGFudHM7IG9yaWdpbmFsIHNlbGVjdG9yLCBubyB2ZW5kb3Iga2VybmVsLiIiIgppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKCkB0cml0b24uaml0CmRlZiBfbmY0X3ZhbHVlKGNvZGUpOgogICAgIyBBbGwgbGl0ZXJhbHMgYXJlIGV4YWN0bHkgdGhlIGNhbm9uaWNhbCBGUDMyIHRhYmxlIHZhbHVlcy4gVHdvIGluZGVwZW5kZW50CiAgICAjIHNlbGVjdGlvbiBjaGFpbnMga2VlcCB0aGUgdGFibGUgdmlzaWJsZSBmb3IgdGhlIHNjYWxhciBzb3VyY2UgdGVzdC4KICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSAwLCAtMS4wLCAtMC42OTYxOTI4MDA5OTg2ODc3KQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDIsIC0wLjUyNTA3MzA1MTQ1MjYzNjcsIG5lZ2F0aXZlKQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDMsIC0wLjM5NDkxNzQ4ODA5ODE0NDUzLCBuZWdhdGl2ZSkKICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSA0LCAtMC4yODQ0NDEzODE2OTI4ODYzNSwgbmVnYXRpdmUpCiAgICBuZWdhdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gNSwgLTAuMTg0NzczNDMwMjI4MjMzMzQsIG5lZ2F0aXZlKQogICAgbmVnYXRpdmUgPSB0bC53aGVyZShjb2RlID09IDYsIC0wLjA5MTA1MDAzNjI1MTU0NDk1LCBuZWdhdGl2ZSkKICAgIG5lZ2F0aXZlID0gdGwud2hlcmUoY29kZSA9PSA3LCAwLjAsIG5lZ2F0aXZlKQogICAgcG9zaXRpdmUgPSB0bC53aGVyZShjb2RlID09IDgsIDAuMDc5NTgwMjk5NTU2MjU1MzQsIDAuMTYwOTMwMjAxNDExMjQ3MjUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTAsIDAuMjQ2MTEyMzAxOTQ1Njg2MzQsIHBvc2l0aXZlKQogICAgcG9zaXRpdmUgPSB0bC53aGVyZShjb2RlID09IDExLCAwLjMzNzkxNTI0MTcxODI5MjI0LCBwb3NpdGl2ZSkKICAgIHBvc2l0aXZlID0gdGwud2hlcmUoY29kZSA9PSAxMiwgMC40NDA3MDk4MjkzMzA0NDQzNCwgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTMsIDAuNTYyNjE3MDAzOTE3Njk0MSwgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTQsIDAuNzIyOTU2ODM2MjIzNjAyMywgcG9zaXRpdmUpCiAgICBwb3NpdGl2ZSA9IHRsLndoZXJlKGNvZGUgPT0gMTUsIDEuMCwgcG9zaXRpdmUpCiAgICByZXR1cm4gdGwud2hlcmUoY29kZSA8IDgsIG5lZ2F0aXZlLCBwb3NpdGl2ZSkK', 'solution/operand_probe.py': 'IiIiUHJlZmxpZ2h0LW9ubHkgZGlyZWN0IFcxNiBvYnNlcnZhdGlvbjsgbm90IHBhcnQgb2YgbWVhc3VyZWQgZm9yd2FyZC4iIiIKaW1wb3J0IHRyaXRvbgppbXBvcnQgdHJpdG9uLmxhbmd1YWdlIGFzIHRsCmZyb20gLm5mNF9mb3JtYXQgaW1wb3J0IF9uZjRfdmFsdWUKCkB0cml0b24uaml0CmRlZiBvcGVyYW5kX3Byb2JlKFBBQ0tFRCxRQUJTLENPREUyLEFCUzIsT0ZGU0VULE9VVCxFOnRsLmNvbnN0ZXhwcixCTE9DSzp0bC5jb25zdGV4cHIpOgogICAgaT10bC5wcm9ncmFtX2lkKDApKkJMT0NLK3RsLmFyYW5nZSgwLEJMT0NLKQogICAgcT10bC5sb2FkKFBBQ0tFRCtpLy8yLGk8RSxvdGhlcj0wKS50byh0bC51aW50MzIpCiAgICBjb2RlPXRsLndoZXJlKChpJjEpPT0wLHE+PjQscSYxNSkKICAgIGJsb2NrPWkvLzY0CiAgICBpbmRleD10bC5sb2FkKFFBQlMrYmxvY2ssaTxFLG90aGVyPTApLnRvKHRsLmludDMyKQogICAgY3Y9dGwubG9hZChDT0RFMitpbmRleCxpPEUsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHNjYWxlMj10bC5sb2FkKEFCUzIrYmxvY2svLzI1NixpPEUsb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgIHByb2R1Y3Q9Y3Yqc2NhbGUyCiAgICBzY2FsZT1wcm9kdWN0K3RsLmxvYWQoT0ZGU0VUKS50byh0bC5mbG9hdDMyKQogICAgdz0oX25mNF92YWx1ZShjb2RlKSpzY2FsZSkudG8odGwuZmxvYXQxNixmcF9kb3duY2FzdF9yb3VuZGluZz0ncnRuZScpCiAgICB0bC5zdG9yZShPVVQraSx3LGk8RSkK', 'solution/parameter_bias.py': 'IiIiTmFycm93IHJhdy1wb2ludGVyIG5vcm1hbGl6YXRpb24gZm9yIG5hdGl2ZSBidWlsdC1pbiBQYXJhbWV0ZXIgYmlhcy4KClJlc2VhcmNoIG9ubHkuIFRoZSBvcmlnaW5hbCBQYXJhbWV0ZXIgcmVtYWlucyB0aGUgaW5wdXQgdG8gbmF0aXZlIE1hdE11bDRCaXQ7Cm9ubHkgaXRzIG5vLWdyYWQgZm9yd2FyZCdzIGtlcm5lbCBvcGVyYW5kIGJlY29tZXMgYSBwbGFpbiBzdG9yYWdlLXNoYXJpbmcgdmlldy4KVGhlcmUgaXMgbm8gY3VzdG9tIGJhY2t3YXJkIGFuZCBubyBnZW5lcmljIHRlbnNvci1zdWJjbGFzcyBhZG1pc3Npb24uCiIiIgppbXBvcnQgaW5zcGVjdAppbXBvcnQgdG9yY2gKCgpkZWYgbm9ybWFsaXplZF9uYXRpdmVfYmlhcyhiaWFzKToKICAgIGlmIGJpYXMgaXMgTm9uZSBvciB0eXBlKGJpYXMpIGlzIHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gYmlhcyxOb25lCiAgICBpZiB0eXBlKGJpYXMpIGlzIG5vdCB0b3JjaC5ubi5QYXJhbWV0ZXI6CiAgICAgICAgcmV0dXJuIE5vbmUsJ2N1c3RvbV9iaWFzX3N1YmNsYXNzJwogICAgaWYgdG9yY2guaXNfZ3JhZF9lbmFibGVkKCk6CiAgICAgICAgcmV0dXJuIE5vbmUsJ3BhcmFtZXRlcl9iaWFzX291dHNpZGVfbmF0aXZlX25vZ3JhZF9mb3J3YXJkJwogICAgaWYgYmlhcy5pc19uZWcoKSBvciBiaWFzLmlzX2NvbmooKSBvciBub3QgYmlhcy5pc19jb250aWd1b3VzKCk6CiAgICAgICAgcmV0dXJuIE5vbmUsJ2xhenlfb3Jfc3RyaWRlZF9wYXJhbWV0ZXJfYmlhcycKICAgIGRldGFjaD10b3JjaC5UZW5zb3IuZGV0YWNoCiAgICBpZiAobm90IGluc3BlY3QuaXNtZXRob2RkZXNjcmlwdG9yKGRldGFjaCkgb3IgZ2V0YXR0cihkZXRhY2gsJ19fb2JqY2xhc3NfXycsTm9uZSkgaXMgbm90IHRvcmNoLl9DLlRlbnNvckJhc2UKICAgICAgICAgICAgb3IgZ2V0YXR0cihkZXRhY2gsJ19fbmFtZV9fJyxOb25lKSE9J2RldGFjaCcgb3IgaGFzYXR0cihkZXRhY2gsJ19fd3JhcHBlZF9fJykpOgogICAgICAgIHJldHVybiBOb25lLCdmb3JlaWduX2RldGFjaF9vd25lcicKICAgIHJhdz1kZXRhY2goYmlhcykKICAgIGlmIHR5cGUocmF3KSBpcyBub3QgdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiBOb25lLCd1bmV4cGVjdGVkX2RldGFjaF9yZXN1bHQnCiAgICByZXR1cm4gcmF3LE5vbmUK', 'solution/reference_forward.py': 'IiIiRnJvemVuIG93biB2NCBrZXJuZWwgcmVmZXJlbmNlIHVzZWQgb25seSBieSBpc29sYXRlZCB0YXJnZXQgcHJlZmxpZ2h0LiIiIgppbXBvcnQgdG9yY2gsIHRyaXRvbgpmcm9tIC5yZWZlcmVuY2Vfa2VybmVsIGltcG9ydCBzb3VwX25mNF9tbWFfbGFuZXByb2JlCgoKZGVmIG9yaWdpbmFsX3JlZmVyZW5jZV9mb3J3YXJkKGRhdGEpOgogICAgYSxwYWNrZWQsc3RhdGUsYmlhcz1kYXRhCiAgICBuLGs9c3RhdGUuc2hhcGU7bT1hLm51bWVsKCkvL2sKICAgIHJlc3VsdD10b3JjaC5lbXB0eSgoKmEuc2hhcGVbOi0xXSxuKSxkZXZpY2U9YS5kZXZpY2UsZHR5cGU9dG9yY2guZmxvYXQxNikKICAgIHNvdXBfbmY0X21tYV9sYW5lcHJvYmVbKHRyaXRvbi5jZGl2KG0sMTYpLHRyaXRvbi5jZGl2KG4sOCkpXSgKICAgICAgICBhLHBhY2tlZCxzdGF0ZS5hYnNtYXgsc3RhdGUuc3RhdGUyLmNvZGUsc3RhdGUuc3RhdGUyLmFic21heCxzdGF0ZS5vZmZzZXQsCiAgICAgICAgYSBpZiBiaWFzIGlzIE5vbmUgZWxzZSBiaWFzLHJlc3VsdCxtLG4sayxiaWFzIGlzIG5vdCBOb25lLAogICAgICAgIG51bV93YXJwcz0xLG51bV9zdGFnZXM9MSxlbmFibGVfZnBfZnVzaW9uPUZhbHNlKQogICAgcmV0dXJuIHJlc3VsdAo=', 'solution/reference_kernel.py': 'IiIiT3JpZ2luYWwgc2luZ2xlLXdhcnAgcmVnaXN0ZXItZnJhZ21lbnQgZmVhc2liaWxpdHkgcHJvYmUsIG5vdCBhbiBpbnN0YWxsZXIuCgpJbnN0cnVjdGlvbiBhbmQgZnJhZ21lbnQgbWFwcGluZyBmcm9tIE5WSURJQSBQVFggSVNBIG0xNm44azggZG9jdW1lbnRhdGlvbiwKbm90IGEgY29waWVkIHZlbmRvciBHRU1NLiBQbGFpbiB0bC5kb3Qgbm8gbG9uZ2VyIHNlbGVjdHMgU003NSBNTUEgaW4gVHJpdG9uMy42LgpUYXJnZXQgcHJlZmxpZ2h0IG11c3QgdmVyaWZ5IDFEIGxhbmUgbGF5b3V0IGFuZCBleGFjdCBmcmFnbWVudCBtYXBwaW5nIGJlZm9yZSB1c2UuCiIiIgppbXBvcnQgdHJpdG9uCmltcG9ydCB0cml0b24ubGFuZ3VhZ2UgYXMgdGwKZnJvbSAubmY0X2Zvcm1hdCBpbXBvcnQgX25mNF92YWx1ZQoKCkB0cml0b24uaml0CmRlZiBfcGFja19oYWxmMihhLCBiKToKICAgIHJldHVybiBhLnRvKHRsLnVpbnQxNiwgYml0Y2FzdD1UcnVlKS50byh0bC51aW50MzIpIHwgKGIudG8odGwudWludDE2LCBiaXRjYXN0PVRydWUpLnRvKHRsLnVpbnQzMikgPDwgMTYpCgoKQHRyaXRvbi5qaXQKZGVmIF9tbWExNjg4KGEwLCBhMSwgYjAsIGMwLCBjMSwgYzIsIGMzKToKICAgIHJldHVybiB0bC5pbmxpbmVfYXNtX2VsZW1lbnR3aXNlKAogICAgICAgICJtbWEuc3luYy5hbGlnbmVkLm0xNm44azgucm93LmNvbC5mMzIuZjE2LmYxNi5mMzIgeyQwLCQxLCQyLCQzfSwgeyQ0LCQ1fSwgeyQ2fSwgeyQ3LCQ4LCQ5LCQxMH07IiwKICAgICAgICBjb25zdHJhaW50cz0iPWYsPWYsPWYsPWYscixyLHIsZixmLGYsZiIsCiAgICAgICAgYXJncz1bYTAsYTEsYjAsYzAsYzEsYzIsYzNdLGR0eXBlPSh0bC5mbG9hdDMyLHRsLmZsb2F0MzIsdGwuZmxvYXQzMix0bC5mbG9hdDMyKSwKICAgICAgICBpc19wdXJlPVRydWUscGFjaz0xKQoKCkB0cml0b24uaml0KGRvX25vdF9zcGVjaWFsaXplPVsnTSddKQpkZWYgc291cF9uZjRfbW1hX2xhbmVwcm9iZShBLCBQQUNLRUQsIFFBQlMsIENPREUyLCBBQlMyLCBPRkZTRVQsIEJpYXMsIE91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICBNLCBOOnRsLmNvbnN0ZXhwciwgSzp0bC5jb25zdGV4cHIsIEJJQVM6dGwuY29uc3RleHByKToKICAgIHRsLnN0YXRpY19hc3NlcnQoSyAlIDY0ID09IDApCiAgICAjIE9uZSB3YXJwICsgMzIgc2NhbGFyIGxvZ2ljYWwgZWxlbWVudHMuIENvZGVnZW4gZ2F0ZSByZWplY3RzIGFub3RoZXIgbGF5b3V0LgogICAgbGFuZT10bC5hcmFuZ2UoMCwzMikKICAgIGdyb3VwPWxhbmUvLzQKICAgIHBhaXI9bGFuZSU0CiAgICBtMD10bC5wcm9ncmFtX2lkKDApKjE2K2dyb3VwCiAgICBuMD10bC5wcm9ncmFtX2lkKDEpKjgrZ3JvdXAKICAgIGMwPXRsLmZ1bGwoKDMyLCksMCx0bC5mbG9hdDMyKTtjMT10bC5mdWxsKCgzMiwpLDAsdGwuZmxvYXQzMikKICAgIGMyPXRsLmZ1bGwoKDMyLCksMCx0bC5mbG9hdDMyKTtjMz10bC5mdWxsKCgzMiwpLDAsdGwuZmxvYXQzMikKICAgIGZvciBibG9jayBpbiB0bC5yYW5nZSgwLEsvLzY0LGxvb3BfdW5yb2xsX2ZhY3Rvcj0xKToKICAgICAgICBzY2FsZV9ibG9jaz1uMCooSy8vNjQpK2Jsb2NrCiAgICAgICAgcWk9dGwubG9hZChRQUJTK3NjYWxlX2Jsb2NrLG4wPE4sb3RoZXI9MCkudG8odGwuaW50MzIpCiAgICAgICAgY3Y9dGwubG9hZChDT0RFMitxaSxuMDxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYWI9dGwubG9hZChBQlMyK3NjYWxlX2Jsb2NrLy8yNTYsbjA8TixvdGhlcj0wKS50byh0bC5mbG9hdDMyKQogICAgICAgIHByb2R1Y3Q9Y3YqYWIKICAgICAgICBzY2FsZT1wcm9kdWN0K3RsLmxvYWQoT0ZGU0VUKS50byh0bC5mbG9hdDMyKQogICAgICAgIGZvciBjaHVuayBpbiB0bC5yYW5nZSgwLDgsbG9vcF91bnJvbGxfZmFjdG9yPTEpOgogICAgICAgICAgICBrMD1ibG9jayo2NCtjaHVuayo4K3BhaXIqMgogICAgICAgICAgICBhMDA9dGwubG9hZChBK20wKksrazAsbTA8TSxvdGhlcj0wKS50byh0bC5mbG9hdDE2KQogICAgICAgICAgICBhMDE9dGwubG9hZChBK20wKksrazArMSxtMDxNLG90aGVyPTApLnRvKHRsLmZsb2F0MTYpCiAgICAgICAgICAgIGExMD10bC5sb2FkKEErKG0wKzgpKksrazAsbTArODxNLG90aGVyPTApLnRvKHRsLmZsb2F0MTYpCiAgICAgICAgICAgIGExMT10bC5sb2FkKEErKG0wKzgpKksrazArMSxtMCs4PE0sb3RoZXI9MCkudG8odGwuZmxvYXQxNikKICAgICAgICAgICAgYnl0ZT10bC5sb2FkKFBBQ0tFRCsobjAqSytrMCkvLzIsbjA8TixvdGhlcj0wKS50byh0bC51aW50MzIpCiAgICAgICAgICAgIGIwPShfbmY0X3ZhbHVlKGJ5dGU+PjQpKnNjYWxlKS50byh0bC5mbG9hdDE2LGZwX2Rvd25jYXN0X3JvdW5kaW5nPSdydG5lJykKICAgICAgICAgICAgYjE9KF9uZjRfdmFsdWUoYnl0ZSYxNSkqc2NhbGUpLnRvKHRsLmZsb2F0MTYsZnBfZG93bmNhc3Rfcm91bmRpbmc9J3J0bmUnKQogICAgICAgICAgICBjMCxjMSxjMixjMz1fbW1hMTY4OChfcGFja19oYWxmMihhMDAsYTAxKSxfcGFja19oYWxmMihhMTAsYTExKSxfcGFja19oYWxmMihiMCxiMSksYzAsYzEsYzIsYzMpCiAgICBjb2w9dGwucHJvZ3JhbV9pZCgxKSo4K3BhaXIqMgogICAgaWYgQklBUzoKICAgICAgICBiaWFzMD10bC5sb2FkKEJpYXMrY29sLGNvbDxOLG90aGVyPTApLnRvKHRsLmZsb2F0MzIpCiAgICAgICAgYmlhczE9dGwubG9hZChCaWFzK2NvbCsxLGNvbCsxPE4sb3RoZXI9MCkudG8odGwuZmxvYXQzMikKICAgICAgICBjMD1jMCtiaWFzMDtjMT1jMStiaWFzMTtjMj1jMitiaWFzMDtjMz1jMytiaWFzMQogICAgdGwuc3RvcmUoT3V0K20wKk4rY29sLGMwLnRvKHRsLmZsb2F0MTYpLCAobTA8TSkmKGNvbDxOKSkKICAgIHRsLnN0b3JlKE91dCttMCpOK2NvbCsxLGMxLnRvKHRsLmZsb2F0MTYpLChtMDxNKSYoY29sKzE8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzgpKk4rY29sLGMyLnRvKHRsLmZsb2F0MTYpLChtMCs4PE0pJihjb2w8TikpCiAgICB0bC5zdG9yZShPdXQrKG0wKzgpKk4rY29sKzEsYzMudG8odGwuZmxvYXQxNiksKG0wKzg8TSkmKGNvbCsxPE4pKQo=', 'solution/web_preflight.py': 'IiIiVGFyZ2V0LW9ubHkgb3JpZ2luYWwgbGFuZSBNTUEgYW5kIHVuY2hhbmdlZCBuYXRpdmUtYmFja3dhcmQgcHJlZmxpZ2h0LiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLG9zLHRyYWNlYmFjawpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKClBST0dSRVNTID0ge30KCmRlZiBydW4oKToKICAgIGlmIG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyk6cmFpc2UgUnVudGltZUVycm9yKCdJbnRlcnByZXRlciBpcyBub3QgYSBkZXZpY2UgZ2F0ZScpCiAgICBpbXBvcnQgdG9yY2gsdHJpdG9uLGJpdHNhbmRieXRlcyBhcyBibmIKICAgIGZyb20gYml0c2FuZGJ5dGVzIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKICAgIGZyb20gYml0c2FuZGJ5dGVzLmF1dG9ncmFkLl9mdW5jdGlvbnMgaW1wb3J0IE1hdE11bDRCaXQKICAgIGZyb20gLiBpbXBvcnQgZ3VhcmRlZF9mb3J3YXJkIGFzIGcKICAgIGZyb20gLmZpeHR1cmVzIGltcG9ydCBFWFBFQ1RFRCxTSEFQRVMsZml4dHVyZSxjYWxsLGV4YWN0LGVuZ2luZWVyaW5nX2Nsb3NlCiAgICBmcm9tIC5vcGVyYW5kX3Byb2JlIGltcG9ydCBvcGVyYW5kX3Byb2JlCiAgICBmcm9tIC5yZWZlcmVuY2VfZm9yd2FyZCBpbXBvcnQgb3JpZ2luYWxfcmVmZXJlbmNlX2ZvcndhcmQKICAgIGlmICh0b3JjaC5fX3ZlcnNpb25fXyE9JzIuMTEuMCtjdTEyOCcgb3IgdHJpdG9uLl9fdmVyc2lvbl9fIT0nMy42LjAnCiAgICAgICAgICAgIG9yIGJuYi5fX3ZlcnNpb25fXyE9JzAuNTAuMicgb3IgdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSE9MQogICAgICAgICAgICBvciB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSE9J1Rlc2xhIFQ0JyBvciB0b3JjaC5jdWRhLmdldF9kZXZpY2VfY2FwYWJpbGl0eSgwKSE9KDcsNSkpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRXhhY3QgdGFyZ2V0IHJ1bnRpbWUgcmVxdWlyZWQnKQogICAgcm93cz1bXTtQUk9HUkVTU1siZm9yd2FyZF9yb3dzIl09cm93cztiYWNrd2FyZD1NYXRNdWw0Qml0LmJhY2t3YXJkCiAgICBmb3IgbSxuIGluIFNIQVBFUzoKICAgICAgICBmb3IgYmlhcyBpbiAoRmFsc2UsVHJ1ZSk6CiAgICAgICAgICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0iZm9yd2FyZF9vcGVyYW5kIixjYXNlPXsiTSI6bSwiTiI6biwiYmlhcyI6Ymlhc30sY29tcGxldGVkX2ZvcndhcmRfcm93cz1sZW4ocm93cykpCiAgICAgICAgICAgIGRhdGE9Zml4dHVyZShtLG4sYmlhcyk7YSxwLHMsYj1kYXRhCiAgICAgICAgICAgIGV4cGVjdGVkPWNhbGwoZGF0YSkKICAgICAgICAgICAgdz1GLmRlcXVhbnRpemVfNGJpdChwLHMpO29ic2VydmVkPXRvcmNoLmVtcHR5X2xpa2UodykKICAgICAgICAgICAgb3BlcmFuZF9wcm9iZVsodHJpdG9uLmNkaXYody5udW1lbCgpLDI1NiksKV0ocCxzLmFic21heCxzLnN0YXRlMi5jb2RlLHMuc3RhdGUyLmFic21heCxzLm9mZnNldCxvYnNlcnZlZCx3Lm51bWVsKCksMjU2LAogICAgICAgICAgICAgICAgbnVtX3dhcnBzPTQsZW5hYmxlX2ZwX2Z1c2lvbj1GYWxzZSkKICAgICAgICAgICAgZXhhY3Qob2JzZXJ2ZWQsdykKICAgICAgICAgICAgUFJPR1JFU1MudXBkYXRlKHN0YWdlPSJjYW5kaWRhdGVfZm9yd2FyZCIsb3BlcmFuZF9mcDE2X2V4YWN0X2Zvcl9jdXJyZW50X2Nhc2U9VHJ1ZSkKICAgICAgICAgICAgYmVmb3JlPWcuQ09VTlRTWydjdXN0b20nXQogICAgICAgICAgICB3aXRoIGcudGVtcG9yYXJ5X25hdGl2ZV9mYWxsYmFjayhFWFBFQ1RFRCk6YWN0dWFsPWNhbGwoZGF0YSkKICAgICAgICAgICAgUFJPR1JFU1MudXBkYXRlKGNvdW50cz1kaWN0KGcuQ09VTlRTKSxmYWxsYmFja19yZWFzb25zPWRpY3QoZy5GQUxMQkFDS19SRUFTT05TKSkKICAgICAgICAgICAgaWYgZy5DT1VOVFNbJ2N1c3RvbSddIT1iZWZvcmUrMTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NhbmRpZGF0ZSBmYWxsYmFjayBpbmFjdGl2ZTsgcmVhc29ucz0nK3JlcHIoZy5GQUxMQkFDS19SRUFTT05TKSkKICAgICAgICAgICAgb3JpZ2luYWw9b3JpZ2luYWxfcmVmZXJlbmNlX2ZvcndhcmQoZGF0YSkKICAgICAgICAgICAgZXhhY3QoYWN0dWFsLG9yaWdpbmFsKQogICAgICAgICAgICBlcnJvcj1lbmdpbmVlcmluZ19jbG9zZShhY3R1YWwsZXhwZWN0ZWQpCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKHsnTSc6bSwnTic6biwnSyc6MTUzNiwnYmlhcyc6Ymlhcywnb3BlcmFuZF9mcDE2X2V4YWN0JzpUcnVlLCdmcm96ZW5fb3JpZ2luYWxfTU1BX291dHB1dF9leGFjdCc6VHJ1ZSwqKmVycm9yfSkKICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0iY29kZWdlbiIsY29tcGxldGVkX2ZvcndhcmRfcm93cz1sZW4ocm93cykpCiAgICBjb2RlZ2VuPVtdO1BST0dSRVNTWyJjb2RlZ2VuIl09Y29kZWdlbgogICAgZm9yIGtleSxjIGluIGcuQ09NUElMRUQuaXRlbXMoKToKICAgICAgICBwdHg9Yy5hc21bJ3B0eCddO2lyPWMuYXNtWyd0dGdpciddCiAgICAgICAgbGF5b3V0cz1bbGluZSBmb3IgbGluZSBpbiBpci5zcGxpdGxpbmVzKCkgaWYgJyN0dGcuYmxvY2tlZDwnIGluIGxpbmVdCiAgICAgICAgZXhwZWN0ZWRfbGF5b3V0PSdzaXplUGVyVGhyZWFkID0gWzFdLCB0aHJlYWRzUGVyV2FycCA9IFszMl0sIHdhcnBzUGVyQ1RBID0gWzRdLCBvcmRlciA9IFswXScKICAgICAgICBpZiAoJy50YXJnZXQgc21fNzUnIG5vdCBpbiBwdHggb3IgJ21tYS5zeW5jLmFsaWduZWQubTE2bjhrOC5yb3cuY29sLmYzMi5mMTYuZjE2LmYzMicgbm90IGluIHB0eAogICAgICAgICAgICAgICAgb3IgcHR4LmNvdW50KCdtbWEuc3luYy5hbGlnbmVkLm0xNm44azgucm93LmNvbC5mMzIuZjE2LmYxNi5mMzInKSE9OAogICAgICAgICAgICAgICAgb3IgJ2ZtYS5ybi5mMzInIGluIHB0eCBvciAnLmZ0eicgaW4gcHR4CiAgICAgICAgICAgICAgICBvciBsZW4obGF5b3V0cykhPTEgb3IgZXhwZWN0ZWRfbGF5b3V0IG5vdCBpbiBsYXlvdXRzWzBdKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdTTTc1IGxhbmUgbGF5b3V0L01NQS9zZXBhcmF0ZS1zY2FsZSBjb2RlZ2VuIGdhdGUgZmFpbGVkJykKICAgICAgICBpZiBjLm5fc3BpbGxzIT0wIG9yIGMubWV0YWRhdGEuc2hhcmVkIT0wOnJhaXNlIFJ1bnRpbWVFcnJvcignVW5leHBlY3RlZCBzcGlsbHMvc2hhcmVkLW1lbW9yeSByZXNvdXJjZSBnYXRlJykKICAgICAgICBjb2RlZ2VuLmFwcGVuZCh7J3JlZ2lzdGVyc19wZXJfdGhyZWFkJzpjLm5fcmVncywnc3BpbGxzJzpjLm5fc3BpbGxzLCdzaGFyZWRfYnl0ZXMnOmMubWV0YWRhdGEuc2hhcmVkLAogICAgICAgICAgICAnbW1hX3N0YXRpY19pbnN0cnVjdGlvbnMnOjgsJ2tleSc6bGlzdChrZXkpLCdwdHhfc2hhMjU2JzpoYXNobGliLnNoYTI1NihwdHguZW5jb2RlKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICAgICAndHRnaXJfc2hhMjU2JzpoYXNobGliLnNoYTI1Nihpci5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksJ21tYSc6VHJ1ZSwnbm9fc2NhbGFyX2ZtYV9vcl9mdHonOlRydWUsJ2xhbmVfbGF5b3V0X3ZlcmlmaWVkJzpUcnVlfSkKICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0ibmF0aXZlX2JhY2t3YXJkIikKICAgIGdyYWRpZW50cz1bXTtQUk9HUkVTU1siZ3JhZGllbnRzIl09Z3JhZGllbnRzCiAgICBmb3IgbiBpbiAoMjU2LDE1MzYpOgogICAgICAgIGRhdGE9Zml4dHVyZSg2NSxuLFRydWUpCiAgICAgICAgZm9yIHNjYWxlIGluICgxLiwxMjguLDY1NTM2Lik6CiAgICAgICAgICAgIGRlZiBvbmUoY3VzdG9tKToKICAgICAgICAgICAgICAgIGEscCxzLGI9ZGF0YTthPWEuZGV0YWNoKCkuY2xvbmUoKS5yZXF1aXJlc19ncmFkXygpO2I9Yi5kZXRhY2goKS5jbG9uZSgpLnJlcXVpcmVzX2dyYWRfKCkKICAgICAgICAgICAgICAgIGN1cnJlbnQ9KGEscCxzLGIpCiAgICAgICAgICAgICAgICBpZiBjdXN0b206CiAgICAgICAgICAgICAgICAgICAgd2l0aCBnLnRlbXBvcmFyeV9uYXRpdmVfZmFsbGJhY2soRVhQRUNURUQpOm91dD1jYWxsKGN1cnJlbnQpCiAgICAgICAgICAgICAgICBlbHNlOm91dD1jYWxsKGN1cnJlbnQpCiAgICAgICAgICAgICAgICAjIEZpeGVkIGNvdGFuZ2VudCBpbmRlcGVuZGVudCBvZiBlaXRoZXIgbnVtZXJpY2FsIGZvcndhcmQgb3V0cHV0LgogICAgICAgICAgICAgICAgdXA9dG9yY2gubGluc3BhY2UoLS4wMSwuMDEsb3V0Lm51bWVsKCksZGV2aWNlPSdjdWRhJykucmVzaGFwZV9hcyhvdXQpLnRvKG91dC5kdHlwZSkqc2NhbGUKICAgICAgICAgICAgICAgIGdhLGdiPXRvcmNoLmF1dG9ncmFkLmdyYWQob3V0LChhLGIpLHVwKQogICAgICAgICAgICAgICAgcmV0dXJuIGdhLGdiCiAgICAgICAgICAgIHJlZmVyZW5jZT1vbmUoRmFsc2UpO2JlZm9yZT1nLkNPVU5UU1snY3VzdG9tJ107YWN0dWFsPW9uZShUcnVlKQogICAgICAgICAgICBpZiBnLkNPVU5UU1snY3VzdG9tJ10hPWJlZm9yZSsxOnJhaXNlIFJ1bnRpbWVFcnJvcignTmF0aXZlIGF1dG9ncmFkIGZvcndhcmQgYnlwYXNzZWQgY2FuZGlkYXRlJykKICAgICAgICAgICAgZm9yIHgseSBpbiB6aXAoYWN0dWFsLHJlZmVyZW5jZSk6ZXhhY3QoeCx5KQogICAgICAgICAgICBncmFkaWVudHMuYXBwZW5kKHsnTic6biwnc2NhbGUnOnNjYWxlLCduYXRpdmVfaW5wdXRfYW5kX2JpYXNfZ3JhZGllbnRzX2V4YWN0JzpUcnVlfSkKICAgICMgTm9uZmluaXRlL3plcm8tY2xhc3MgY2hlY2tzIGNhbm5vdCBiZSByZXBsYWNlZCBieSB0b2xlcmFuY2VzLgogICAgUFJPR1JFU1MudXBkYXRlKHN0YWdlPSJjbGFzc2VzX2FuZF9iaWFzX3dpdG5lc3MiKQogICAgY2xhc3Nlcz1bXTtQUk9HUkVTU1siY2xhc3NfZml4dHVyZXMiXT1jbGFzc2VzCiAgICBmb3IgbW9kZSBpbiAoJ3plcm8nLCduYW4nLCdwb3NpbmYnLCduZWdpbmYnKToKICAgICAgICBkYXRhPWxpc3QoZml4dHVyZSg2NSwyNTYsRmFsc2UpKTtkYXRhWzBdLnplcm9fKCkKICAgICAgICBpZiBtb2RlIT0nemVybyc6ZGF0YVswXVs6LDBdPWRpY3QobmFuPWZsb2F0KCduYW4nKSxwb3NpbmY9ZmxvYXQoJ2luZicpLG5lZ2luZj0tZmxvYXQoJ2luZicpKVttb2RlXQogICAgICAgIGV4cGVjdGVkPWNhbGwoZGF0YSkKICAgICAgICB3aXRoIGcudGVtcG9yYXJ5X25hdGl2ZV9mYWxsYmFjayhFWFBFQ1RFRCk6YWN0dWFsPWNhbGwoZGF0YSkKICAgICAgICBpZiBtb2RlPT0nemVybyc6ZXhhY3QoYWN0dWFsLGV4cGVjdGVkKQogICAgICAgIGVsc2U6ZW5naW5lZXJpbmdfY2xvc2UoYWN0dWFsLGV4cGVjdGVkKQogICAgICAgIGNsYXNzZXMuYXBwZW5kKG1vZGUpCiAgICAjIENhbmRpZGF0ZSBhcml0aG1ldGljLXN0YWdlIHdpdG5lc3MuIE5hdGl2ZSBDVURBIGVwaWxvZ3VlIGlzIG5vdCBhc3N1bWVkLgogICAgUFJPR1JFU1MudXBkYXRlKHN0YWdlPSJjYW5kaWRhdGVfYmlhc19zdGFnZSIpCiAgICB3PXRvcmNoLm9uZXMoMjU2LDE1MzYsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KQogICAgcCxzPUYucXVhbnRpemVfNGJpdCh3LGJsb2Nrc2l6ZT02NCxjb21wcmVzc19zdGF0aXN0aWNzPVRydWUscXVhbnRfdHlwZT0nbmY0JykKICAgIGE9dG9yY2guemVyb3MoNjUsMTUzNixkZXZpY2U9J2N1ZGEnLGR0eXBlPXRvcmNoLmZsb2F0MTYpO2FbOiwwXT0xLjthWzosMV09LjAwMDQKICAgIGI9dG9yY2guZnVsbCgoMjU2LCksLjAwMDIsZGV2aWNlPSdjdWRhJyxkdHlwZT10b3JjaC5mbG9hdDE2KTtkYXRhPShhLHAscyxiKQogICAgZXhwZWN0ZWQ9Y2FsbChkYXRhKQogICAgd2l0aCBnLnRlbXBvcmFyeV9uYXRpdmVfZmFsbGJhY2soRVhQRUNURUQpOmFjdHVhbD1jYWxsKGRhdGEpCiAgICBmcm9tIC5iaWFzX3N0YWdlX2dhdGUgaW1wb3J0IHZlcmlmeV9iaWFzX3N0YWdlCiAgICBkZWNvZGVkPUYuZGVxdWFudGl6ZV80Yml0KHAscykKICAgIGZwMzJfcmVmZXJlbmNlPXRvcmNoLm5uLmZ1bmN0aW9uYWwubGluZWFyKGEuZmxvYXQoKSxkZWNvZGVkLmZsb2F0KCkpCiAgICBuYXRpdmVfd2l0aG91dF9iaWFzPWNhbGwoKGEscCxzLE5vbmUpKQogICAgYmlhc19zdGFnZT12ZXJpZnlfYmlhc19zdGFnZShhY3R1YWwsZXhwZWN0ZWQsZnAzMl9yZWZlcmVuY2UsYixuYXRpdmVfd2l0aG91dF9iaWFzKQogICAgYmlhc19zdGFnZVsnc291cmNlX2JvdW5kX3JlYXNvbl9mb3JfcmV2aXNlZF9nYXRlJ109anNvbi5sb2FkcygoUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCdiaWFzX2dhdGVfcHJvdmVuYW5jZS5qc29uJykpLnJlYWRfdGV4dCgpKQogICAgUFJPR1JFU1NbJ2JpYXNfc3RhZ2UnXT1iaWFzX3N0YWdlCiAgICAjIFJlYWwgbmF0aXZlIG1vZHVsZXMvUGFyYW1ldGVycyBtdXN0IGFjdGl2YXRlIGJlZm9yZSB0aW1pbmcgaXMgYWxsb3dlZC4KICAgIGZyb20gLm1vZHVsZV9wcmVmbGlnaHQgaW1wb3J0IHJ1biBhcyBtb2R1bGVfcnVuLCBQUk9HUkVTUyBhcyBNT0RVTEVfUFJPR1JFU1MKICAgIFBST0dSRVNTLnVwZGF0ZShzdGFnZT0iYWN0dWFsX25hdGl2ZV9MaW5lYXI0Yml0Iixtb2R1bGVfcHJvZ3Jlc3M9TU9EVUxFX1BST0dSRVNTKQogICAgbW9kdWxlX2NoZWNrcz1tb2R1bGVfcnVuKCkKICAgIFBST0dSRVNTWydtb2R1bGVfY2hlY2tzJ109bW9kdWxlX2NoZWNrcwogICAgaWYgTWF0TXVsNEJpdC5iYWNrd2FyZCBpcyBub3QgYmFja3dhcmQ6cmFpc2UgUnVudGltZUVycm9yKCdOYXRpdmUgYmFja3dhcmQgaWRlbnRpdHkgY2hhbmdlZCcpCiAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkKICAgIHJldHVybiB7J3Jvd3MnOnJvd3MsJ2NvZGVnZW4nOmNvZGVnZW4sJ2dyYWRpZW50cyc6Z3JhZGllbnRzLCdjbGFzc19maXh0dXJlcyc6Y2xhc3NlcywKICAgICAgICAnYmlhc19zdGFnZSc6Ymlhc19zdGFnZSwnbW9kdWxlX2NoZWNrcyc6bW9kdWxlX2NoZWNrcywnY291bnRzJzpkaWN0KGcuQ09VTlRTKSwKICAgICAgICAnbmF0aXZlX2JhY2t3YXJkX2lkZW50aXR5X3ByZXNlcnZlZCc6VHJ1ZSwndGFza19xdWFsaXR5JzonVU5LTk9XTid9CgpkZWYgbWFpbigpOgogICAgcGFyc2VyPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cGFyc2VyLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2FyZ3M9cGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcmVwb3J0PXsnc3RhdHVzJzonRkFJTEVEX1BSRUZMSUdIVF9PTkxZJywnc2NvcGUnOidNSUNST0JFTkNIX09OTFknLCd0cmFpbmluZ19leGVjdXRlZCc6RmFsc2UsJ3Njb3JlJzpOb25lLCdhZG1pc3Npb24nOidVTktOT1dOJ30KICAgIHRyeTpyZXBvcnRbJ2NoZWNrcyddPXJ1bigpO3JlcG9ydFsnc3RhdHVzJ109J1BBU1NFRF9QUkVGTElHSFRfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6cmVwb3J0LnVwZGF0ZShlcnJvcj1yZXByKGUpLHRyYWNlYmFjaz10cmFjZWJhY2suZm9ybWF0X2V4YygpLHByb2dyZXNzPWRpY3QoUFJPR1JFU1MpKQogICAgUGF0aChhcmdzLm91dCkud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpCiAgICBwcmludChqc29uLmR1bXBzKHsnc3RhdHVzJzpyZXBvcnRbJ3N0YXR1cyddLCdvdXRwdXQnOmFyZ3Mub3V0fSksZmx1c2g9VHJ1ZSkKICAgIHJhaXNlIFN5c3RlbUV4aXQocmVwb3J0WydzdGF0dXMnXSE9J1BBU1NFRF9QUkVGTElHSFRfT05MWScpCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'reviewed_support.py': 'aW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixtYXRoLG9zLHNodXRpbCxzeXMsdGhyZWFkaW5nLHRpbWUsdXVpZCx6aXBmaWxlCgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBwcm9jZXNzX2d1YXJkCgppbXBvcnQgb3duZWRfcHJvY2Vzc190cmVlIGFzIHRyZWUKCmRlZiBzaGEocCk6cmV0dXJuIGhhc2hsaWIuc2hhMjU2KFBhdGgocCkucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKZGVmIHJlYWQocCk6cmV0dXJuIGpzb24ubG9hZHMoUGF0aChwKS5yZWFkX3RleHQoKSkKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKZGVmIHZlcmlmeV9vZmZpY2lhbChyZXBvLG1hbmlmZXN0KToKIGlmIG5vdCByZXBvLmlzX2RpcigpIG9yIHJlcG8uaXNfc3ltbGluaygpOnJhaXNlIFJ1bnRpbWVFcnJvcignU09VUF9SRVBPIG11c3QgYmUgYSByZWFsIHByZXBhcmVkIGpvYiBkaXJlY3RvcnknKQogZm9yIG5hbWUsd2FudGVkIGluIG1hbmlmZXN0Lml0ZW1zKCk6CiAgcD1yZXBvL25hbWUKICBpZiBub3QgcC5pc19maWxlKCkgb3IgcC5pc19zeW1saW5rKCkgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgUnVudGltZUVycm9yKCdQcm90ZWN0ZWQgb2ZmaWNpYWwgc291cmNlIG1pc21hdGNoOiAnK25hbWUpCiBhY3R1YWw9e3N0cihwLnJlbGF0aXZlX3RvKHJlcG8pKSBmb3IgZm9sZGVyIGluICgnaGFybmVzcycsJ3Rhc2snLCd0ZXN0cycpIGZvciBwIGluIChyZXBvL2ZvbGRlcikucmdsb2IoJyonKQogICAgICAgICBpZiBwLmlzX2ZpbGUoKSBhbmQgJ19fcHljYWNoZV9fJyBub3QgaW4gcC5wYXJ0cyBhbmQgcC5zdWZmaXggbm90IGluICgnLnB5YycsJy5weW8nKX0KIGlmIGFjdHVhbCE9c2V0KG1hbmlmZXN0KTpyYWlzZSBSdW50aW1lRXJyb3IoJ1VuZXhwZWN0ZWQgcHJvdGVjdGVkIG9mZmljaWFsIGZpbGVzJykKCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgoKRVhQT1JUX09NSVRURUQ9e30KCmRlZiBleHBvcnRfc21hbGwocnVuLGV4cG9ydCk6CiAiIiJSZWFkLW9ubHkgaW5jcmVtZW50YWwgZXhwb3J0OyBubyBhZGFwdGVycywgd2VpZ2h0cyBvciBjb21waWxlciBjYWNoZXMuIiIiCiB1c2VkPXN1bShwLnN0YXQoKS5zdF9zaXplIGZvciBwIGluIGV4cG9ydC5yZ2xvYignKicpIGlmIHAuaXNfZmlsZSgpKQogZXhjbHVkZWQ9eydhZGFwdGVyJywndHJpdG9uX2NhY2hlJywnaW5kdWN0b3JfY2FjaGUnLCdwcmVmbGlnaHRfdHJpdG9uJywncHJlZmxpZ2h0X2luZHVjdG9yJywnb3JpZ2luYWxfc29sdXRpb24nfQogZm9yIGJhc2UsZGlycyxmaWxlcyBpbiBvcy53YWxrKHJ1bik6CiAgZGlyc1s6XT1bZCBmb3IgZCBpbiBkaXJzIGlmIGQgbm90IGluIGV4Y2x1ZGVkIGFuZCBub3QgZC5zdGFydHN3aXRoKCdjaGVja3BvaW50LScpIGFuZCBub3QgKFBhdGgoYmFzZSkvZCkuaXNfc3ltbGluaygpXQogIGZvciBuYW1lIGluIGZpbGVzOgogICBwYXRoPVBhdGgoYmFzZSkvbmFtZTtyZWw9cGF0aC5yZWxhdGl2ZV90byhydW4pCiAgIHRyeToKICAgIGlmIG5vdCBwYXRoLmlzX3N5bWxpbmsoKSBhbmQgcGF0aC5zdWZmaXggaW4gKCcuanNvbicsJy5qc29ubCcsJy5sb2cnLCcueWFtbCcpOgogICAgIGlmIHBhdGguc3RhdCgpLnN0X3NpemU+PTQqMTAyNCoqMjoKICAgICAgRVhQT1JUX09NSVRURURbc3RyKHJlbCldPSdwZXJfZmlsZV9saW1pdF80TWlCJztjb250aW51ZQogICAgIGRlc3Q9ZXhwb3J0LydyYXcnL3JlbDtkZXN0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgpjbGFzcyBFeHBvcnRlcjoKIGRlZiBfX2luaXRfXyhzZWxmLHJ1bixleHBvcnQpOnNlbGYucnVuPXJ1bjtzZWxmLmV4cG9ydD1leHBvcnQ7c2VsZi5zdG9wPXRocmVhZGluZy5FdmVudCgpO3NlbGYudGhyZWFkPU5vbmUKIGRlZiBfX2VudGVyX18oc2VsZik6CiAgZGVmIGxvb3AoKToKICAgd2hpbGUgbm90IHNlbGYuc3RvcC53YWl0KDEpOmV4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKICBzZWxmLnRocmVhZD10aHJlYWRpbmcuVGhyZWFkKHRhcmdldD1sb29wLGRhZW1vbj1UcnVlKTtzZWxmLnRocmVhZC5zdGFydCgpO3JldHVybiBzZWxmCiBkZWYgX19leGl0X18oc2VsZiwqZXhjKTpzZWxmLnN0b3Auc2V0KCk7c2VsZi50aHJlYWQuam9pbigpO2V4cG9ydF9zbWFsbChzZWxmLnJ1bixzZWxmLmV4cG9ydCkKCmRlZiBpZGxlX2NoZWNrKGVudixyZXBvLHJ1bixsb2NrZmQpOgogY29kZT0nJydpbXBvcnQganNvbixzdWJwcm9jZXNzLHN5cwpwPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktY29tcHV0ZS1hcHBzPXBpZCx1c2VkX2dwdV9tZW1vcnknLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCmFzc2VydCBub3QgcC5zdGRvdXQuc3RyaXAoKSwnR1BVIHByb2Nlc3NlcyBzdGlsbCBhY3RpdmUgYWZ0ZXIgcHJvYmUvcHJlZmxpZ2h0JwpyPXN1YnByb2Nlc3MucnVuKFsnbnZpZGlhLXNtaScsJy0tcXVlcnktZ3B1PWluZGV4LG1lbW9yeS51c2VkJywnLS1mb3JtYXQ9Y3N2LG5vaGVhZGVyLG5vdW5pdHMnXSxjYXB0dXJlX291dHB1dD1UcnVlLHRleHQ9VHJ1ZSxjaGVjaz1UcnVlKQpyb3dzPVtsaW5lLnNwbGl0KCcsJykgZm9yIGxpbmUgaW4gci5zdGRvdXQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KYXNzZXJ0IHJvd3MgYW5kIHN1bShpbnQoeFsxXSkgZm9yIHggaW4gcm93cyk8MTAwMCwnR1BVIHBoeXNpY2FsIGludmVudG9yeSBub3QgaWRsZScKcHJpbnQoanNvbi5kdW1wcyh7J2lkbGUnOlRydWUsJ3BoeXNpY2FsX21lbW9yeV9yb3dzJzpyb3dzfSkpJycnCiByZXR1cm4gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctYycsY29kZV0sY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydpZGxlLmxvZycsdGltZW91dD0yMCxsb2NrX2ZkPWxvY2tmZCkK', 'worker.py': 'IiIiRnJlc2gtcHJvY2VzcyBxL3Ytc2hhcGVkIGZvcndhcmQgbWljcm9iZW5jaCwgbmV2ZXIgYSB0cmFpbmluZyBiZW5jaG1hcmsuIiIiCmltcG9ydCBhcmdwYXJzZSxjb250ZXh0bGliLGpzb24sb3MsdGltZSx0cmFjZWJhY2sKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpkZWYgbWFpbigpOgogICAgcD1hcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpO3AuYWRkX2FyZ3VtZW50KCctLW1ldGhvZCcsY2hvaWNlcz1bJ25hdGl2ZScsJ2NhbmRpZGF0ZSddLHJlcXVpcmVkPVRydWUpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2FyZ3M9cC5wYXJzZV9hcmdzKCk7b3V0PVBhdGgoYXJncy5vdXQpCiAgICBpZiBvdXQuZXhpc3RzKCk6cmFpc2UgRmlsZUV4aXN0c0Vycm9yKG91dCkKICAgIHJlcG9ydD17J3N0YXR1cyc6J0ZBSUxFRF9NSUNST0JFTkNIX09OTFknLCdzY29wZSc6J01JQ1JPQkVOQ0hfT05MWScsJ21ldGhvZCc6YXJncy5tZXRob2QsCiAgICAgICAgJ2dwdV9leGVjdXRlZCc6RmFsc2UsJ3RyYWluaW5nX2V4ZWN1dGVkJzpGYWxzZSwnY29tcGxldGVfZml4ZWRfdGFzayc6RmFsc2UsJ3Jvd3MnOltdLAogICAgICAgICdzY29yZSc6Tm9uZSwnYWRtaXNzaW9uJzonVU5LTk9XTicsJ21ldGhvZF92YXJpYW50cyc6eyduYXRpdmUnOidibmIgbmF0aXZlIGRpc3BhdGNoZXIvZmFsbGJhY2snLAogICAgICAgICAgICAnY2FuZGlkYXRlJzonc2FtZSBuYXRpdmUgZGlzcGF0Y2hlciB3aXRoIG9yaWdpbmFsNjR4NjQgZm91ci13YXJwIHJlZ2lzdGVyLXJldXNlIE5GNCBNTUEgZmFsbGJhY2snfSwKICAgICAgICAnbWVhc3VyZW1lbnRfY29udHJhY3QnOnsnY29sZCc6J3N5bmNocm9uaXplZCBmaXJzdCB3cmFwcGVyIGNhbGwgcGVyIHNoYXBlOyBpbmNsdWRlcyBuZWVkZWQgSklULCBhbGxvY2F0aW9uLCBkaXNwYXRjaCBhbmQgZGV2aWNlIHdvcmssIG5vdCBwdXJlIGNvbXBpbGUnLAogICAgICAgICAgICAnd2FybSc6JzEyIHNlcGFyYXRlIENVREEgZXZlbnQgc3RyZWFtIGludGVydmFscyBhZnRlciAzIHdhcm11cCBjYWxsczsgUHl0aG9uIGxhdW5jaCBnYXBzIGluY2x1ZGVkJywKICAgICAgICAgICAgJ29yZGVyJzonbmF0aXZlIHRoZW4gY2FuZGlkYXRlIGluIHNlcGFyYXRlIHByb2Nlc3NlcyBhbmQgcHJpdmF0ZSBjYWNoZXM7IG5vdCBjb3VudGVyYmFsYW5jZWQnLAogICAgICAgICAgICAnaW5wdXRzJzonc3ludGhldGljIHNjYWxlZCByYW5kb20gcS92IHNoYXBlczsgbm90IGNhcHR1cmVkIHRhc2sgYWN0aXZhdGlvbiBkaXN0cmlidXRpb25zJywKICAgICAgICAgICAgJ25vX3Rhc2tfc3BlZWRfb3Jfc2hhcmVfaW5mZXJlbmNlJzpUcnVlfX0KICAgIHRyeToKICAgICAgICBpZiBvcy5lbnZpcm9uLmdldCgnVFJJVE9OX0lOVEVSUFJFVCcpOnJhaXNlIFJ1bnRpbWVFcnJvcignSW50ZXJwcmV0ZXIgZm9yYmlkZGVuJykKICAgICAgICBjYWNoZT1QYXRoKG9zLmVudmlyb25bJ1RSSVRPTl9DQUNIRV9ESVInXSkKICAgICAgICBpZiBub3QgY2FjaGUuaXNfZGlyKCkgb3IgYW55KGNhY2hlLml0ZXJkaXIoKSk6cmFpc2UgUnVudGltZUVycm9yKCdQcml2YXRlIGVtcHR5IGluaXRpYWwgY2FjaGUgcmVxdWlyZWQnKQogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gc29sdXRpb24uZml4dHVyZXMgaW1wb3J0IEVYUEVDVEVELFNIQVBFUyxmaXh0dXJlLGNhbGwsZW5naW5lZXJpbmdfY2xvc2UKICAgICAgICBmcm9tIHNvbHV0aW9uIGltcG9ydCBndWFyZGVkX2ZvcndhcmQgYXMgZwogICAgICAgIGN0eD1sYW1iZGE6Zy50ZW1wb3JhcnlfbmF0aXZlX2ZhbGxiYWNrKEVYUEVDVEVEKSBpZiBhcmdzLm1ldGhvZD09J2NhbmRpZGF0ZScgZWxzZSBjb250ZXh0bGliLm51bGxjb250ZXh0KCkKICAgICAgICBmb3IgbSxuIGluIFNIQVBFUzoKICAgICAgICAgICAgZGF0YT1maXh0dXJlKG0sbixUcnVlKTtiZWZvcmU9Zy5DT1VOVFNbJ2N1c3RvbSddCiAgICAgICAgICAgIHdpdGggY3R4KCk6CiAgICAgICAgICAgICAgICB0b3JjaC5jdWRhLnN5bmNocm9uaXplKCk7c3RhcnQ9dGltZS5wZXJmX2NvdW50ZXIoKTthY3R1YWw9Y2FsbChkYXRhKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpO2NvbGQ9dGltZS5wZXJmX2NvdW50ZXIoKS1zdGFydAogICAgICAgICAgICBleHBlY3RlZD1jYWxsKGRhdGEpO251bWVyaWM9ZW5naW5lZXJpbmdfY2xvc2UoYWN0dWFsLGV4cGVjdGVkKQogICAgICAgICAgICBkZWwgYWN0dWFsLGV4cGVjdGVkCiAgICAgICAgICAgIGlmIGFyZ3MubWV0aG9kPT0nY2FuZGlkYXRlJyBhbmQgZy5DT1VOVFNbJ2N1c3RvbSddIT1iZWZvcmUrMTpyYWlzZSBSdW50aW1lRXJyb3IoJ0NvbGQgY2FuZGlkYXRlIGluYWN0aXZlJykKICAgICAgICAgICAgcGFpcnM9W10KICAgICAgICAgICAgd2l0aCBjdHgoKToKICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKDMpOmNhbGwoZGF0YSkKICAgICAgICAgICAgICAgIHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKQogICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMTIpOgogICAgICAgICAgICAgICAgICAgIHN0YXJ0PXRvcmNoLmN1ZGEuRXZlbnQoZW5hYmxlX3RpbWluZz1UcnVlKTtzdG9wPXRvcmNoLmN1ZGEuRXZlbnQoZW5hYmxlX3RpbWluZz1UcnVlKQogICAgICAgICAgICAgICAgICAgIHN0YXJ0LnJlY29yZCgpO2NhbGwoZGF0YSk7c3RvcC5yZWNvcmQoKTtwYWlycy5hcHBlbmQoKHN0YXJ0LHN0b3ApKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5zeW5jaHJvbml6ZSgpCiAgICAgICAgICAgIHNhbXBsZXM9W2EuZWxhcHNlZF90aW1lKGIpIGZvciBhLGIgaW4gcGFpcnNdCiAgICAgICAgICAgIGlmIGFyZ3MubWV0aG9kPT0nY2FuZGlkYXRlJyBhbmQgZy5DT1VOVFNbJ2N1c3RvbSddIT1iZWZvcmUrMTY6cmFpc2UgUnVudGltZUVycm9yKCdXYXJtIGNhbmRpZGF0ZSBpbmFjdGl2ZScpCiAgICAgICAgICAgIHJvdz17J00nOm0sJ04nOm4sJ0snOjE1MzYsJ2JpYXMnOlRydWUsJ2NvbGRfZmlyc3Rfd3JhcHBlcl93YWxsX3MnOmNvbGQsCiAgICAgICAgICAgICAgICAgJ3dhcm1fZm9yd2FyZF9jdWRhX2V2ZW50X21zJzpzYW1wbGVzLCdudW1lcmljcyc6bnVtZXJpYywnY291bnRzJzpkaWN0KGcuQ09VTlRTKX0KICAgICAgICAgICAgcmVwb3J0Wydyb3dzJ10uYXBwZW5kKHJvdyk7b3V0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsaW5kZW50PTIsYWxsb3dfbmFuPUZhbHNlKSsnXG4nKQogICAgICAgIGlmIGxlbihyZXBvcnRbJ3Jvd3MnXSkhPTY6cmFpc2UgUnVudGltZUVycm9yKCdJbmNvbXBsZXRlIGZpeHR1cmUgc2V0JykKICAgICAgICByZXBvcnQudXBkYXRlKHN0YXR1cz0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScsZ3B1X2V4ZWN1dGVkPVRydWUpCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBlOnJlcG9ydC51cGRhdGUoZXJyb3I9cmVwcihlKSx0cmFjZWJhY2s9dHJhY2ViYWNrLmZvcm1hdF9leGMoKSkKICAgIG91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yLGFsbG93X25hbj1GYWxzZSkrJ1xuJykKICAgIHByaW50KGpzb24uZHVtcHMoeydzdGF0dXMnOnJlcG9ydFsnc3RhdHVzJ10sJ21ldGhvZCc6YXJncy5tZXRob2QsJ3Jvd3MnOmxlbihyZXBvcnRbJ3Jvd3MnXSl9KSxmbHVzaD1UcnVlKQogICAgcmFpc2UgU3lzdGVtRXhpdChyZXBvcnRbJ3N0YXR1cyddIT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScpCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'driver.py': 'IiIiQm91bmRlZCBuYXRpdmUvY2FuZGlkYXRlIG9yaWdpbmFsIE5GNCBmb3J3YXJkIE1JQ1JPQkVOQ0hfT05MWSBvcmNoZXN0cmF0aW9uLgoKTmV2ZXIgaW5zdGFsbHMgaW50byBTT1VQX1JFUE8gb3IgY2FsbHMgYSB0cmFpbmluZyBoYXJuZXNzLiBFeGlzdGluZyBzb3VyY2UvcnVudGltZSwKbG9jaywgcGlkZmQvc3VicmVhcGVyLCBxdWFyYW50aW5lIGFuZCBpZGxlLWNoZWNrIGhlbHBlcnMgYXJlIHByZXNlcnZlZCB1bmNoYW5nZWQuClRoZSBkZXBsb3llZCBzZXJ2aWNlJ3MgbGltaXRzIHJlbWFpbiBhdXRob3JpdGF0aXZlOyBlYWNoIEdQVSBjaGlsZCBnZXRzIDw9ODAgcy4KIiIiCmltcG9ydCBmY250bCwgaGFzaGxpYiwganNvbiwgb3MsIHN5cywgdGltZSwgdXVpZApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHByb2Nlc3NfZ3VhcmQKaW1wb3J0IG93bmVkX3Byb2Nlc3NfdHJlZSBhcyB0cmVlCmZyb20gcmV2aWV3ZWRfc3VwcG9ydCBpbXBvcnQgdmVyaWZ5X29mZmljaWFsLCBpZGxlX2NoZWNrLCBFeHBvcnRlciwgZXhwb3J0X3NtYWxsLCByZWFkLCB3cml0ZSwgc2hhCgpMT0NLID0gUGF0aCgnL3RtcC9zb3VwLWdwdTAubG9jaycpCk1BUktFUiA9IFBhdGgoJy90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCcpCgpkZWYgbWFpbigpOgogICAgaGVyZSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKICAgIGlmIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVBPJykgb3Igbm90IG9zLmVudmlyb24uZ2V0KCdTT1VQX1JFU1VMVFMnKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdSZXF1aXJlcyBkZXBsb3llZCBTT1VQX1JFUE8vU09VUF9SRVNVTFRTLCBubyBtYW51YWwgZmFsbGJhY2snKQogICAgcmVwbywgcmVzdWx0cyA9IFBhdGgob3MuZW52aXJvblsnU09VUF9SRVBPJ10pLmFic29sdXRlKCksIFBhdGgob3MuZW52aXJvblsnU09VUF9SRVNVTFRTJ10pLmFic29sdXRlKCkKICAgIGlmIHJlcG8uaXNfc3ltbGluaygpIG9yIHJlc3VsdHMuaXNfc3ltbGluaygpIG9yIHJlcG8gPT0gcmVzdWx0czoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCdQcmVwYXJlZCBjaGVja291dC9yZXN1bHRzIG11c3QgYmUgc2VwYXJhdGUgcmVhbCBkaXJlY3RvcmllcycpCiAgICBhc3NlcnQgUGF0aCgnL2thZ2dsZS93b3JraW5nJykuaXNfZGlyKCkKICAgIHJlc3VsdHMubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBpZGVudGl0eSA9ICduZjQtbW1hLW1pY3JvYmVuY2gtb25seS0nICsgdXVpZC51dWlkNCgpLmhleFs6MTJdCiAgICBydW4sIGV4cG9ydCA9IFBhdGgoJy9rYWdnbGUvd29ya2luZycpL2lkZW50aXR5LCByZXN1bHRzL2lkZW50aXR5CiAgICBydW4ubWtkaXIoZXhpc3Rfb2s9RmFsc2UpOyBleHBvcnQubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgICBzdGF0dXMgPSBkaWN0KHN0YXR1cz0nUFJFUEFSSU5HX01JQ1JPQkVOQ0hfT05MWScsc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScsCiAgICAgICAgICAgICAgICAgIHRyYWluaW5nX2V4ZWN1dGVkPUZhbHNlLGNvbXBsZXRlX2ZpeGVkX3Rhc2s9RmFsc2Usc2NvcmU9Tm9uZSxhZG1pc3Npb249J1VOS05PV04nLAogICAgICAgICAgICAgICAgICBwZXJfZ3B1X2NoaWxkX3dhbGxfY2FwX3M9ODAsY2FwX2J5cGFzcz1GYWxzZSwKICAgICAgICAgICAgICAgICAgY2FuZGlkYXRlX3NvdXJjZV9zaGEyNTY9cmVhZChoZXJlLydzb3VyY2VfbWFuaWZlc3QuanNvbicpLAogICAgICAgICAgICAgICAgICBtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlcj1bJ25hdGl2ZScsJ2NhbmRpZGF0ZSddLHByb2Nlc3Nlcz17fSkKICAgIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgIGxvY2tmZCA9IE5vbmU7IHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICB0cnk6CiAgICAgICAgZm9yIG5hbWUsZGlnZXN0IGluIHJlYWQoaGVyZS8ncGF5bG9hZF9tYW5pZmVzdC5qc29uJykuaXRlbXMoKToKICAgICAgICAgICAgcCA9IGhlcmUvbmFtZQogICAgICAgICAgICBhc3NlcnQgbm90IHAuaXNfc3ltbGluaygpIGFuZCBwLmlzX2ZpbGUoKSBhbmQgc2hhKHApPT1kaWdlc3QsICdQYXlsb2FkIGNoYW5nZWQ6ICcrbmFtZQogICAgICAgIGlmIE1BUktFUi5leGlzdHMoKTogcmFpc2UgUnVudGltZUVycm9yKCdHUFUgcXVhcmFudGluZSByZXF1aXJlcyBvcGVyYXRvciByZWNvbmNpbGlhdGlvbicpCiAgICAgICAgbG9ja2ZkID0gb3Mub3BlbihMT0NLLG9zLk9fQ1JFQVR8b3MuT19SRFdSfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICAgICAgZmNudGwuZmxvY2sobG9ja2ZkLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICAgICAgICBpZiBNQVJLRVIuZXhpc3RzKCk6IHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgYXBwZWFyZWQgd2hpbGUgbG9ja2luZycpCiAgICAgICAgdHJlZS5wcmVwYXJlKCk7IHRyZWUuYmVnaW5fcXVhcmFudGluZShNQVJLRVIpCiAgICAgICAgb2ZmaWNpYWwgPSByZWFkKGhlcmUvJ29mZmljaWFsX21hbmlmZXN0Lmpzb24nKQogICAgICAgIHZlcmlmeV9vZmZpY2lhbChyZXBvLG9mZmljaWFsKQogICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbixIRl9IVUJfT0ZGTElORT0nMScsSEZfREFUQVNFVFNfT0ZGTElORT0nMScsVFJBTlNGT1JNRVJTX09GRkxJTkU9JzEnLAogICAgICAgICAgICAgICAgICAgUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnLFBZVEhPTlVOQlVGRkVSRUQ9JzEnKQogICAgICAgIGVudi5wb3AoJ1NPVVBfUk9UQVJZX0NPREVHRU5fRElSJyxOb25lKQogICAgICAgIHdpdGggRXhwb3J0ZXIocnVuLGV4cG9ydCk6CiAgICAgICAgICAgIHByb2JlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3J1bnRpbWVfcHJvYmUucHknKSxzdHIocnVuLydydW50aW1lX3Byb2JlLmpzb24nKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPXJlcG8sZW52PWVudixsb2c9cnVuLydydW50aW1lX3Byb2JlLmxvZycsdGltZW91dD00MCxsb2NrX2ZkPWxvY2tmZCkKICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVsncnVudGltZV9wcm9iZSddID0gcHJvYmUKICAgICAgICAgICAgaWYgcHJvYmVbJ2V4aXRfY29kZSddIG9yIG5vdCByZWFkKHJ1bi8ncnVudGltZV9wcm9iZS5qc29uJykuZ2V0KCdwYXNzZWQnKToKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignRXhhY3QgZGVwbG95ZWQgcnVudGltZSBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgZGVmIGlkbGUobGFiZWwpOgogICAgICAgICAgICAgICAgZm9sZGVyPXJ1bi8oJ2lkbGVfJytsYWJlbCk7Zm9sZGVyLm1rZGlyKGV4aXN0X29rPUZhbHNlKQogICAgICAgICAgICAgICAgcmVzdWx0PWlkbGVfY2hlY2soZW52LHJlcG8sZm9sZGVyLGxvY2tmZCkKICAgICAgICAgICAgICAgIHN0YXR1c1sncHJvY2Vzc2VzJ11bJ2lkbGVfJytsYWJlbF09cmVzdWx0CiAgICAgICAgICAgICAgICBpZiByZXN1bHRbJ2V4aXRfY29kZSddOnJhaXNlIFJ1bnRpbWVFcnJvcignSWRsZSBnYXRlIGZhaWxlZDogJytsYWJlbCkKICAgICAgICAgICAgaWRsZSgnYmVmb3JlX3ByZWZsaWdodCcpCiAgICAgICAgICAgICMgUmV2aXNlZCBzb3VyY2UtYm91bmQgbnVtZXJpY2FsL1BUWCBhbmQgYWN0dWFsIExpbmVhcjRiaXQgZ2F0ZXMsIGlzb2xhdGVkIGZyb20gbWVhc3VyZWQgY2FjaGVzLgogICAgICAgICAgICBwcmVlbnYgPSBkaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihydW4vJ3ByZWZsaWdodF90cml0b24nKSwKICAgICAgICAgICAgICAgICAgICAgICAgICBUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfaW5kdWN0b3InKSkKICAgICAgICAgICAgcHJlID0gcHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsJy1tJywnc29sdXRpb24ud2ViX3ByZWZsaWdodCcsJy0tb3V0JyxzdHIocnVuLydwcmVmbGlnaHQuanNvbicpXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY3dkPWhlcmUsZW52PXByZWVudixsb2c9cnVuLydwcmVmbGlnaHQubG9nJyx0aW1lb3V0PTgwLGxvY2tfZmQ9bG9ja2ZkKQogICAgICAgICAgICBzdGF0dXNbJ3Byb2Nlc3NlcyddWydwcmVmbGlnaHQnXSA9IHByZQogICAgICAgICAgICBpZiBwcmVbJ2V4aXRfY29kZSddIG9yIHJlYWQocnVuLydwcmVmbGlnaHQuanNvbicpLmdldCgnc3RhdHVzJykhPSdQQVNTRURfUFJFRkxJR0hUX09OTFknOgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdORjQgTU1BIG51bWVyaWNhbC9QVFgvYWN0dWFsLW1vZHVsZSBnYXRlIGZhaWxlZDsgbm8gYmVuY2htYXJrJykKICAgICAgICAgICAgc3RhdHVzWydzdGF0dXMnXT0nUlVOTklOR19NSUNST0JFTkNIX09OTFknO3dyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN0YXR1cykKICAgICAgICAgICAgaWRsZSgncHJlZmxpZ2h0JykKICAgICAgICAgICAgZm9yIG1ldGhvZCBpbiBzdGF0dXNbJ21ldGhvZHNfaW5fZXhlY3V0aW9uX29yZGVyJ106CiAgICAgICAgICAgICAgICBjYWNoZT1ydW4vJ3RyaXRvbl9jYWNoZScvbWV0aG9kO2NhY2hlLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1GYWxzZSkKICAgICAgICAgICAgICAgIGNoaWxkZW52PWRpY3QoZW52LFRSSVRPTl9DQUNIRV9ESVI9c3RyKGNhY2hlKSxUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydpbmR1Y3Rvcl9jYWNoZScvbWV0aG9kKSkKICAgICAgICAgICAgICAgIHByb2Nlc3M9cHJvY2Vzc19ndWFyZC5ydW4oW3N5cy5leGVjdXRhYmxlLCctQicsc3RyKGhlcmUvJ3dvcmtlci5weScpLCctLW1ldGhvZCcsbWV0aG9kLCctLW91dCcsc3RyKHJ1bi8obWV0aG9kKycuanNvbicpKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGN3ZD1oZXJlLGVudj1jaGlsZGVudixsb2c9cnVuLyhtZXRob2QrJy5sb2cnKSx0aW1lb3V0PTgwLGxvY2tfZmQ9bG9ja2ZkKQogICAgICAgICAgICAgICAgc3RhdHVzWydwcm9jZXNzZXMnXVttZXRob2RdPXByb2Nlc3MKICAgICAgICAgICAgICAgIGlmIHByb2Nlc3NbJ2V4aXRfY29kZSddIG9yIHJlYWQocnVuLyhtZXRob2QrJy5qc29uJykpLmdldCgnc3RhdHVzJykhPSdQQVNTRURfTUlDUk9CRU5DSF9PTkxZJzoKICAgICAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0luY29tcGxldGUvZmFpbGVkL2NlbnNvcmVkIG1pY3JvYmVuY2htYXJrOiAnK21ldGhvZCkKICAgICAgICAgICAgICAgIGlkbGUobWV0aG9kKQogICAgICAgICAgICAgICAgd3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3RhdHVzKQogICAgICAgICAgICB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxvZmZpY2lhbCkKICAgICAgICAgICAgcmVwb3J0cz17bWV0aG9kOnJlYWQocnVuLyhtZXRob2QrJy5qc29uJykpIGZvciBtZXRob2QgaW4gc3RhdHVzWydtZXRob2RzX2luX2V4ZWN1dGlvbl9vcmRlciddfQogICAgICAgICAgICBhc3NlcnQgYWxsKGxlbihyWydyb3dzJ10pPT02IGFuZCByWyd0cmFpbmluZ19leGVjdXRlZCddIGlzIEZhbHNlIGFuZCByWydjb21wbGV0ZV9maXhlZF90YXNrJ10gaXMgRmFsc2UgZm9yIHIgaW4gcmVwb3J0cy52YWx1ZXMoKSkKICAgICAgICAgICAgd3JpdGUocnVuLydjb21wYXJpc29uLmpzb24nLGRpY3Qoc2NvcGU9J01JQ1JPQkVOQ0hfT05MWScscmVwb3J0cz1yZXBvcnRzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNjb3JlPU5vbmUsYWRtaXNzaW9uPSdVTktOT1dOJyxjb21wbGV0ZV9maXhlZF90YXNrPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5vdGU9J1JhdyBmaXJzdC1jYWxsIHdhbGwgYW5kIHdhcm0gQ1VEQS1ldmVudCBzYW1wbGVzLCBub3QgcHVyZSBjb21waWxlIGNvc3Qgb3IgdGFzayBzcGVlZHVwOyBubyByYXRpb3Mgb3Igb3duZXJzaGlwLXNoYXJlIGluZmVyZW5jZS4nKSkKICAgICAgICAgICAgc3RhdHVzWydzdGF0dXMnXT0nUEFTU0VEX01JQ1JPQkVOQ0hfT05MWScKICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBzdGF0dXMudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9GQUlMRURfT1JfQ0VOU09SRURfTUlDUk9CRU5DSF9PTkxZJyxlcnJvcj1yZXByKGV4YykpCiAgICAgICAgaWYgbG9ja2ZkIGlzIG5vdCBOb25lOnRyZWUucXVhcmFudGluZSgnTkY0IG1pY3JvYmVuY2htYXJrIGZhaWxlZDogJytyZXByKGV4YykpCiAgICBmaW5hbGx5OgogICAgICAgIHN0YXR1c1snb3JjaGVzdHJhdG9yX3dhbGxfcyddPXRpbWUubW9ub3RvbmljKCktc3RhcnRlZAogICAgICAgIGV4cG9ydF9zbWFsbChydW4sZXhwb3J0KTt3cml0ZShleHBvcnQvJ3N0YXR1cy5qc29uJyxzdGF0dXMpCiAgICAgICAgaWYgbG9ja2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICB0cmVlLmZpbmlzaF9xdWFyYW50aW5lKCk7b3MuY2xvc2UobG9ja2ZkKQogICAgcHJpbnQoanNvbi5kdW1wcyh7J3Njb3BlJzonTUlDUk9CRU5DSF9PTkxZJywnc3RhdHVzJzpzdGF0dXNbJ3N0YXR1cyddLCdleHBvcnQnOnN0cihleHBvcnQpLAogICAgICAgICAgICAgICAgICAgICAgJ3Njb3JlJzpOb25lLCdhZG1pc3Npb24nOidVTktOT1dOJ30pLGZsdXNoPVRydWUpCiAgICByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgc3RhdHVzWydzdGF0dXMnXT09J1BBU1NFRF9NSUNST0JFTkNIX09OTFknIGVsc2UgMSkKCmlmIF9fbmFtZV9fPT0nX19tYWluX18nOm1haW4oKQo=', 'source_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJiNDkzZjE3ZDgzMDhmYzVjYmUyZGIwN2Q2ZWZjYjE1OGIxMDkwMzViMzI0N2UxNGU3NzZlMjU3YzY1ZDQzYTQxIiwKICAic29sdXRpb24vYmlhc19nYXRlX3Byb3ZlbmFuY2UuanNvbiI6ICIyYWI2NjkzNTM2MTI3MTk0NmY2Yjk0N2ZmZTgzNjBiZTZiMmI4ZjhhMDkzZGVkYTZkNGIzY2Q0ZTljODE2MzdhIiwKICAic29sdXRpb24vYmlhc19zdGFnZV9nYXRlLnB5IjogImVmYmFjNGI4ZjM3MjUwZWVhOTExNWU0MTcwNzc2MmZiMDNjOWQ4MzU5MmI3MGJjNjRiZjc0NmZmYzNkN2JkYzAiLAogICJzb2x1dGlvbi9maXh0dXJlcy5weSI6ICJiNmZlMDY2ZDMxM2MyOTAxNjc3NWVmNjBjODRhZmE2MzgxNjMwODU2OWU0NjZlOTQxMGEyM2FlZGYxM2YzYzUwIiwKICAic29sdXRpb24vZ3VhcmRlZF9mb3J3YXJkLnB5IjogIjQ4ZDdkNmFhZjM3ZDdkNmRhZTkzM2E0Njk2YTQxMTUyNzgyYTAzODhhYTczMzZiNTJhNzFiODlkYzVkOTk5ZmQiLAogICJzb2x1dGlvbi9rZXJuZWwucHkiOiAiM2ZjNzcxOTE3OTA0YTA1ZDlkNGE5YWRlZmY4M2YwNzk0YWE0ZjVkN2RhMTBmNzA4MjZiZmI4NzRiZjA0YzNjZiIsCiAgInNvbHV0aW9uL21vZHVsZV9wcmVmbGlnaHQucHkiOiAiOGQ1MzFkODJlMTk1MDQyY2VjYzcxZTYyM2RjZjlkMzI5YjM5YWFjMzg0MTlmZmQ4N2NhMmE5ZTBjMDc0MzU3MyIsCiAgInNvbHV0aW9uL25hdGl2ZV9zb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI5MDViODgwNjM5NDFjMDM5YWUzMGYxNDdjNTI0ODMyYWFkNzc2NmU5OWViMWM0NTE5OThiYWM0MzQxN2I2YWM3IiwKICAic29sdXRpb24vbmY0X2Zvcm1hdC5weSI6ICIyMTE4OWY5N2E3NDUyYTk2MDdlMjg5MWJlMDQwMmNkYjQ1NWI3MjE5MTljZDkxZGFlZGRhMTZjMWQ3YzhiMGU0IiwKICAic29sdXRpb24vb3BlcmFuZF9wcm9iZS5weSI6ICI1NDZhNjJhNmQwYmI4YmY3ZDk1YmVlNDM0ZGE1NWQ4ZTExMTFjZThlNDViZmNkNzgyZmI2NGY4MDg3Mzg4NTI4IiwKICAic29sdXRpb24vcGFyYW1ldGVyX2JpYXMucHkiOiAiZTVhYzQwMjIwMTI0ODc2MzI3MzQzOTJkMzgyNDk5OTE5MzAyZjM0MzhmMjMyYmRjNTgxNjlmOTkyZjBjZGRjNyIsCiAgInNvbHV0aW9uL3JlZmVyZW5jZV9mb3J3YXJkLnB5IjogIjc2NWQxZmU5NmU5YTY5NThkNmNkZTdkOTk4OWVjY2IzNDE4ZDhjYTI3YTQ5OGFiZWUwNDQyMzRlMzk1MGUxNDAiLAogICJzb2x1dGlvbi9yZWZlcmVuY2Vfa2VybmVsLnB5IjogImIwMzc5NWYxZTRmZWFhOTliNzhkZWUzZDFiOGY4YzFlYWNhMzc5NDNhODJlZGMzZWYxNDAwZTFjYTRjYzFjZjgiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjQ5ZTExNDdlMjI0NDcwZjYwOTAyNTAxODljZTExYWExMWVlMjk4ZTliZmIyYjk0ODA1OWYxZjY3NWMxNmE4N2YiCn0K', 'payload_manifest.json': 'ewogICJzb2x1dGlvbi9fX2luaXRfXy5weSI6ICJiNDkzZjE3ZDgzMDhmYzVjYmUyZGIwN2Q2ZWZjYjE1OGIxMDkwMzViMzI0N2UxNGU3NzZlMjU3YzY1ZDQzYTQxIiwKICAic29sdXRpb24vYmlhc19nYXRlX3Byb3ZlbmFuY2UuanNvbiI6ICIyYWI2NjkzNTM2MTI3MTk0NmY2Yjk0N2ZmZTgzNjBiZTZiMmI4ZjhhMDkzZGVkYTZkNGIzY2Q0ZTljODE2MzdhIiwKICAic29sdXRpb24vYmlhc19zdGFnZV9nYXRlLnB5IjogImVmYmFjNGI4ZjM3MjUwZWVhOTExNWU0MTcwNzc2MmZiMDNjOWQ4MzU5MmI3MGJjNjRiZjc0NmZmYzNkN2JkYzAiLAogICJzb2x1dGlvbi9maXh0dXJlcy5weSI6ICJiNmZlMDY2ZDMxM2MyOTAxNjc3NWVmNjBjODRhZmE2MzgxNjMwODU2OWU0NjZlOTQxMGEyM2FlZGYxM2YzYzUwIiwKICAic29sdXRpb24vZ3VhcmRlZF9mb3J3YXJkLnB5IjogIjQ4ZDdkNmFhZjM3ZDdkNmRhZTkzM2E0Njk2YTQxMTUyNzgyYTAzODhhYTczMzZiNTJhNzFiODlkYzVkOTk5ZmQiLAogICJzb2x1dGlvbi9rZXJuZWwucHkiOiAiM2ZjNzcxOTE3OTA0YTA1ZDlkNGE5YWRlZmY4M2YwNzk0YWE0ZjVkN2RhMTBmNzA4MjZiZmI4NzRiZjA0YzNjZiIsCiAgInNvbHV0aW9uL21vZHVsZV9wcmVmbGlnaHQucHkiOiAiOGQ1MzFkODJlMTk1MDQyY2VjYzcxZTYyM2RjZjlkMzI5YjM5YWFjMzg0MTlmZmQ4N2NhMmE5ZTBjMDc0MzU3MyIsCiAgInNvbHV0aW9uL25hdGl2ZV9zb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI5MDViODgwNjM5NDFjMDM5YWUzMGYxNDdjNTI0ODMyYWFkNzc2NmU5OWViMWM0NTE5OThiYWM0MzQxN2I2YWM3IiwKICAic29sdXRpb24vbmY0X2Zvcm1hdC5weSI6ICIyMTE4OWY5N2E3NDUyYTk2MDdlMjg5MWJlMDQwMmNkYjQ1NWI3MjE5MTljZDkxZGFlZGRhMTZjMWQ3YzhiMGU0IiwKICAic29sdXRpb24vb3BlcmFuZF9wcm9iZS5weSI6ICI1NDZhNjJhNmQwYmI4YmY3ZDk1YmVlNDM0ZGE1NWQ4ZTExMTFjZThlNDViZmNkNzgyZmI2NGY4MDg3Mzg4NTI4IiwKICAic29sdXRpb24vcGFyYW1ldGVyX2JpYXMucHkiOiAiZTVhYzQwMjIwMTI0ODc2MzI3MzQzOTJkMzgyNDk5OTE5MzAyZjM0MzhmMjMyYmRjNTgxNjlmOTkyZjBjZGRjNyIsCiAgInNvbHV0aW9uL3JlZmVyZW5jZV9mb3J3YXJkLnB5IjogIjc2NWQxZmU5NmU5YTY5NThkNmNkZTdkOTk4OWVjY2IzNDE4ZDhjYTI3YTQ5OGFiZWUwNDQyMzRlMzk1MGUxNDAiLAogICJzb2x1dGlvbi9yZWZlcmVuY2Vfa2VybmVsLnB5IjogImIwMzc5NWYxZTRmZWFhOTliNzhkZWUzZDFiOGY4YzFlYWNhMzc5NDNhODJlZGMzZWYxNDAwZTFjYTRjYzFjZjgiLAogICJzb2x1dGlvbi93ZWJfcHJlZmxpZ2h0LnB5IjogIjQ5ZTExNDdlMjI0NDcwZjYwOTAyNTAxODljZTExYWExMWVlMjk4ZTliZmIyYjk0ODA1OWYxZjY3NWMxNmE4N2YiLAogICJwcm9jZXNzX2d1YXJkLnB5IjogIjgyNDY1OTQzYTNhOWFmMzgzMWE1Mzk5YTNhNTVkMDlhYzI5MGJhNzM2NWM0ZTgwZDZiYzVlNzlmYWI2MmE2ZmYiLAogICJvd25lZF9wcm9jZXNzX3RyZWUucHkiOiAiNjkzZGRlNzA4ZGE1YzA2MDUzYWYyMzk0NjM5NTIxMTVlZjVlODg2NWY1MGJjYmM1MWEyMDM4MjM3Y2E1N2Q2OSIsCiAgInJ1bnRpbWVfcHJvYmUucHkiOiAiZWZlMWI3Y2YzZjNjZjFlMzdiYmMxNjlhODQ2ODg1OWM1MTI3ZWMzMjgxYTM3ZTM5MDY4NzAzYjJlYjNjMTQ4MCIsCiAgIm9mZmljaWFsX21hbmlmZXN0Lmpzb24iOiAiZDAxM2UxNjMzOTQwYzI3ZDcwOGUyNDc0YTgyODA5OTAwMDBlMzMwMThlODg5ZWQxNDY1NWM4NTYzMjlmZjFkOCIsCiAgInJldmlld2VkX3N1cHBvcnQucHkiOiAiN2M1ZDYwMmQwNzEzODNlZjA3ZmRkZDk5NjcwZWQxY2NlMzE0MWZlYWJkNWFmODM5NGYwNTNjNmI0ZTA3MTIyYyIsCiAgIndvcmtlci5weSI6ICI1YWFiOTFkNDdmM2E5NTRhOTlhZDhhYmIxYzFkM2I0MDYzOTk2MDkyN2Q0NjgwYWZlZGY3NTNiZWFkYmE2YTEyIiwKICAiZHJpdmVyLnB5IjogImEyNjU0OWE5ZDI2NTE4NGZmMDk2ZTA5ZGM5N2MzNzQwNzQzNWYyYjgwMGM0MTc1OWRkYzFmNWMxMGVjMzJkZGIiLAogICJzb3VyY2VfbWFuaWZlc3QuanNvbiI6ICI5ZWJhNDc0OGFmNmYwN2ZmZWM4YTk3NGNkZGFmYzI1Yzg2NDgyZDkwYTQ0ZmFjNjQ2NmNlYzM4ZDg5YjEzNmY2Igp9Cg=='}
PAYLOAD_SHA256 = {'solution/__init__.py': 'b493f17d8308fc5cbe2db07d6efcb158b109035b3247e14e776e257c65d43a41', 'solution/bias_gate_provenance.json': '322592c88b775dd48a54a25907491a66312d399d3519f4ccf64d0b9fef867b5e', 'solution/bias_stage_gate.py': 'efbac4b8f37250eea9115e41707762fb03c9d83592b70bc64bf746ffc3d7bdc0', 'solution/fixtures.py': 'b6fe066d313c29016775ef60c84afa63816308569e466e9410a23aedf13f3c50', 'solution/guarded_forward.py': '48d7d6aaf37d7d6dae933a4696a41152782a0388aa7336b52a71b89dc5d999fd', 'solution/kernel.py': '3fc771917904a05d9d4a9adeff83f0794aa4f5d7da10f70826bfb874bf04c3cf', 'solution/module_preflight.py': '8d531d82e195042cecc71e623dcf9d329b39aac38419ffd87ca2a9e0c0743573', 'solution/native_source_manifest.json': '905b88063941c039ae30f147c524832aad7766e99eb1c451998bac43417b6ac7', 'solution/nf4_format.py': '21189f97a7452a9607e2891be0402cdb455b721919cd91daedda16c1d7c8b0e4', 'solution/operand_probe.py': '546a62a6d0bb8bf7d95bee434da55d8e1111ce8e45bfcd782fb64f8087388528', 'solution/parameter_bias.py': 'e5ac4022012487632734392d382499919302f3438f232bdc58169f992f0cddc7', 'solution/reference_forward.py': '765d1fe96e9a6958d6cde7d9989eccb3418d8ca27a498abee044234e3950e140', 'solution/reference_kernel.py': 'b03795f1e4feaa99b78dee3d1b8f8c1eaca37943a82edc3ef1400e1ca4cc1cf8', 'solution/web_preflight.py': '49e1147e224470f6090250189ce11aa11ee298e9bfb2b948059f1f675c16a87f', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'reviewed_support.py': '7c5d602d071383ef07fddd99670ed1cce3141feabd5af8394f053c6b4e07122c', 'worker.py': '5aab91d47f3a954a99ad8abb1c1d3b40639960927d4680afedf753beadba6a12', 'driver.py': 'a26549a9d265184ff096e09dc97c37407435f2b800c41759ddc1f5c10ec32ddb', 'source_manifest.json': '9eba4748af6f07ffec8a974cddafc25c86482d90a44fac6466cec38d89b136f6', 'payload_manifest.json': '21d36f29fa99b224a3b177a19b721dea69e45b42ac7fd29e7114e52834d578cc'}
STAGED=Path(tempfile.mkdtemp(prefix='nf4-mma-microbench-',dir='/kaggle/working'))
for name,encoded in PAYLOAD.items():
    parts=Path(name).parts
    assert not Path(name).is_absolute() and '..' not in parts
    raw=base64.b64decode(encoded,validate=True)
    assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256[name]
    target=STAGED/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
print('Prepared MICROBENCH_ONLY payload:',STAGED)


In [ ]:
import subprocess,sys,signal
process=subprocess.Popen([sys.executable,'-B',str(STAGED/'driver.py')],start_new_session=True)
try:
    return_code=process.wait()
except BaseException:
    if process.poll() is None:process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try:process.wait()
        except KeyboardInterrupt:pass
    raise
if return_code:raise RuntimeError('MICROBENCH_ONLY failed or censored; inspect exported status. No automatic retry.')
